# Prithvi-EO-2.0 flood mapping — DIMER E2E segmentation fine-tuning tutorial (standalone)

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/prithvi-flood-segmentation-pipeline) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/prithvi-flood-segmentation-pipeline/blob/main/tutorials/prithvi_flood_segmentation_colab.ipynb) [![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-ibm--nasa--geospatial%2FPrithvi--EO--2.0--300M--TL--Sen1Floods11-ffcc4d?style=flat)](https://huggingface.co/ibm-nasa-geospatial/Prithvi-EO-2.0-300M-TL-Sen1Floods11) [![Upstream](https://img.shields.io/badge/Upstream-NASA--IMPACT%2FPrithvi--EO--2.0-181717?style=flat&logo=github&logoColor=white)](https://github.com/NASA-IMPACT/Prithvi-EO-2.0) [![Paper](https://img.shields.io/badge/arXiv-2412.02732-b31b1b.svg)](https://arxiv.org/abs/2412.02732)

**Profile:** `E2E`  
**Mode:** `GUIDED`  
**Notebook specification:** DIMER Notebook Specification 2.0 — **standalone** (§4)  
**Capability:** flood-extent segmentation of six-band Sentinel-2 chips with a Prithvi-EO-2.0 ViT-L encoder and UPerNet decoder, held-out IoU/F1 against a no-water baseline, and bounded fine-tuning of the decoder to labelled chips

**This notebook is standalone.** It carries the repository's package (3 modules under `src/prithvi_flood_segmentation_pipeline/`, at revision `uncommitted`) verbatim in Section 2, the pinned model identity and the per-file SHA-256 manifest in Section 3, and the exact runtime pins in Section 1, so it keeps working after export even if the repository changes or disappears. Its only external dependencies are the pinned PyPI distributions and the Hugging Face Hub at the immutable revision `91ce9d38086a80b078a192b374df758b8855b732` (~1284 MB, digest-verified before loading). It was generated by `tools/build_notebook.py` (build_notebook.py/2); edit the repository and regenerate rather than editing cells.

**Run all:** Selecting **Run all** in a fresh **GPU** runtime installs the pinned dependencies (torch, torchvision, terratorch and its stack, tifffile, numpy, safetensors, huggingface-hub), stages and digest-verifies the pinned Prithvi flood checkpoint (1.28 GB) from the Hub, statically audits the Lightning pickle against an allow-list, converts it once into safetensors with a pinned digest, rebuilds the architecture from the installed `terratorch` package and loads it strictly, fetches 44 hand-labelled Sen1Floods11 chips (104 MB of digest-verified GeoTIFFs, no credential), validates them and assigns the official roles (24 training, 8 validation, 12 test), segments the held-out chips with the frozen model and scores them against the no-water baseline, runs a bounded fine-tuning of the neck, decoder and head, scores the same chips again, segments the three example chips shipped with the upstream repository, exports the adapter as safetensors with a manifest, and reloads that artifact into a fresh pipeline to verify prediction parity. The default path needs no repository clone, no DIMER worker or service, no credential, no upload dialog and no configuration edit (NOTEBOOK_SPEC 2.0 §5). On a T4 the whole path takes a few minutes of model time after the downloads; the terratorch install is the slowest step.

**Bring Your Own Data:** After the tutorial workflow completes, set `USE_BYOD = True` in Section 4 and re-run from that cell to supply your own labelled chips as a zip holding `pairs.csv` (columns `id`, `image`, `label`) beside six-band 512 × 512 GeoTIFF chips (blue, green, red, narrow NIR, SWIR 1, SWIR 2 — reflectance in [0, 1] or × 10 000) and single-band label rasters (0 = no water, 1 = water, −1 = no data); at least four chips with some water. Your chips are split by seed into training, validation and test sets and flow through the same contract — validation, frozen baseline, adaptation, held-out evaluation, inference, artifact export and reload parity. The expected schema, the ceilings and the privacy guidance are stated in the Prerequisites and in Section 4, and uploaded files stay inside this runtime. BYOD is optional and never part of the default path.

Prithvi-EO-2.0 (Szwarcman et al., 2024) is NASA and IBM's foundation model for Harmonized Landsat Sentinel-2 imagery: a ViT-L masked autoencoder pretrained on 4.2 M global multispectral samples, here in its `TL` variant with temporal and location embeddings. The checkpoint packaged here is the upstream authors' fine-tune for flood mapping — the 300 M-parameter encoder, a learned pyramid neck over four encoder depths, a UPerNet decoder and a two-class head — trained on the 446 hand-labelled Sentinel-2 chips of Sen1Floods11 with TerraTorch.

Two things about this row are handled in the open. **The upstream asset is a pickle** — a PyTorch Lightning checkpoint. Section 3 downloads and digest-verifies it, statically lists every global the pickle would import (a state dict of tensors and nothing else), refuses anything outside that allow-list, unpickles it exactly once through torch's weights-only loader, and writes a safetensors file whose digest is pinned in the carried module; the model you run is rebuilt from the installed `terratorch` package and loads that file strictly. **The model is already fine-tuned on this dataset**, so the bounded adaptation in Section 6 is a demonstration of the contract on chips it has seen the distribution of, selected by validation loss with the frozen model as epoch 0; the honest number is the paired held-out comparison in Section 7, and the point of the contract is the same recipe applied to *your* labelled chips from another sensor, season or region.

**Learning objectives:** install the pinned runtime; inspect the carried pipeline, dataset and metrics modules; stage and digest-verify a pickled checkpoint, read its static audit and see it converted into safetensors; fetch and validate real labelled multispectral chips with an ignore class; read pixel IoU, F1, precision and recall against a no-water baseline; run a bounded decoder fine-tuning with explicit hyperparameters and frozen BatchNorm statistics; compare the adapted and frozen models on the same held-out chips; segment new chips; and export a safetensors adapter that reloads against the pinned base with verified parity.

**This notebook does not demonstrate:** the Sentinel-1 (SAR) route of Sen1Floods11, the temporal and location embeddings (the packaged fine-tune ran without metadata, and so does this pipeline), tiling of scenes larger than 512 × 512, atmospheric correction, cloud masking, the published benchmark scores, and any claim that a 44-chip sample stands in for an operational evaluation. The repository exposes none of these.

## Prerequisites

- **Runtime:** a fresh supported **GPU** runtime (Google Colab T4 or better, or a Jupyter kernel with a CUDA GPU and Python 3.12): the ViT-L encoder runs in float16 autocast and the default adaptation needs about 2.5 GB of GPU memory; on CPU one 512 × 512 chip takes tens of seconds and the adaptation would take an hour. About 3 GB of disk is needed for the checkpoint and its conversion; the `terratorch` install pulls torchgeo, lightning and their dependencies and takes several minutes.
- **Knowledge:** what a multispectral reflectance chip is (bands, scaling, no-data), what a pixel-wise segmentation mask and an ignore class are, and how IoU, precision and recall are read against a majority baseline.
- **Executable serialization handled explicitly:** the pinned checkpoint is a pickle. It is digest-verified, statically audited against an allow-list (audit digest pinned) and unpickled **once** through torch's weights-only loader to produce the safetensors the model is actually loaded from. No Hub-hosted Python module is imported; `terratorch` is installed from PyPI at a pinned version.
- **Data contract:** a record is `{{id, image, label}}` — a (6, 512, 512) reflectance array (or a GeoTIFF; 13-band Sentinel-2 L1C files are reduced to the six Prithvi bands) with values in [0, 1] or × 10 000, no-data 0 or −9999, and a (512, 512) mask with 0 / 1 / −1. Validation is structural: nothing checks that the bands are the right six in the right order, that the reflectance is corrected, or that the label belongs to the chip.
- **Privacy:** Do not upload confidential or restricted data to a hosted runtime unless you are authorized to process it there — commercial imagery under licence or unreleased disaster assessments are exactly that. The default path uploads nothing.
- **External access (data):** besides the Hub, the default path fetches 88 pinned objects (44 chips and their labels, about 104 MB) from the public Sen1Floods11 bucket `storage.googleapis.com/sen1floods11` over HTTPS, digest-verified before decoding; Sen1Floods11 is CC BY 4.0 (Cloud to Street).
- **External access:** the Hugging Face Hub only, to fetch the pinned `ibm-nasa-geospatial/Prithvi-EO-2.0-300M-TL-Sen1Floods11` snapshot (~1284 MB in total) at revision `91ce9d38086a…`. No GitHub access and no credentials are required; nothing is installed from this repository.

## 1. Install the pinned runtime

The dependency set is pinned exactly (the same `==` pins as the repository's `pyproject.toml` at the generating revision) and installed directly — there is no repository clone and no package install. If a pin replaces a distribution this runtime has already imported, the cell stops with a restart instruction rather than continuing with mixed versions. Look for a dictionary reporting the notebook's source revision, Python, `torch`, `timm`, `lightning`, `tifffile` versions, and whether CUDA is available.

In [ ]:
import importlib
import importlib.metadata
import os
import platform
import subprocess
import sys

PINS = [
    'torch==2.14.0',
    'torchvision==0.29.0',
    'terratorch==1.2.13',
    'tifffile==2026.9.15',
    'numpy==2.5.3',
    'safetensors==0.8.0',
    'huggingface-hub==1.32.0',
]
NOTEBOOK_SOURCE = {
    'repository': 'prithvi-flood-segmentation-pipeline',
    'repository_revision': 'uncommitted',
    'embedded_module': 'src/prithvi_flood_segmentation_pipeline/pipeline.py',
    'embedded_modules': ['src/prithvi_flood_segmentation_pipeline/pipeline.py', 'src/prithvi_flood_segmentation_pipeline/metrics.py', 'src/prithvi_flood_segmentation_pipeline/samples.py'],
    'module_sha256': 'bc6fc4e6a608fbeb26a69d3578ff8e2785130ce69e7b44c269276275dfe13e37',
    'generator': 'build_notebook.py/2',
    'notebook_spec': '2.0',
}
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'

def _installed_version(distribution):
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None

if not SKIP_INSTALL:
    # Capture every distribution already imported in this runtime, whatever its module name
    # (PIL -> pillow), so a pinned install that replaces a loaded package is detected and the
    # notebook stops with a restart instruction instead of continuing with mixed versions.
    _module_dists = importlib.metadata.packages_distributions()
    _loaded = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded}
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINS], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

import torch, timm, lightning, tifffile
print({'notebook_source': NOTEBOOK_SOURCE, 'python': platform.python_version(), 'torch': torch.__version__, 'timm': timm.__version__, 'lightning': lightning.__version__, 'tifffile': tifffile.__version__, 'cuda': torch.cuda.is_available()})

## 2. Pipeline code (carried verbatim from `src/prithvi_flood_segmentation_pipeline/` @ `uncommitted`)

The next 3 cell(s) **are** the repository's package, module by module in dependency order: the pinned identity constants, snapshot verification (`verify_snapshot`), staged download (`stage_missing_files`), the named operational ceilings, the public validation and evaluation helpers, and the pipeline class. The text is the modules', byte for byte, except for the rewrite rules listed in `tools/build_notebook.py` (1 rule(s), plus the removal of package-relative `from .x import` lines, whose names are already defined by the preceding cells). The repository's parity test (`tests/test_notebook_parity.py`) fails whenever these cells and the modules diverge, so what you run here is what the repository tests. Nothing in these cells runs a model yet.

**Module 1/3:** `src/prithvi_flood_segmentation_pipeline/pipeline.py`

In [ ]:
"""Prithvi-EO-2.0-300M-TL Sen1Floods11 (`ibm-nasa-geospatial/Prithvi-EO-2.0-300M-TL-Sen1Floods11`) DIMER
pipeline: verified snapshot, one-time conversion of the pickled Lightning checkpoint into safetensors, flood-extent
segmentation of six-band Sentinel-2 chips, held-out evaluation against a no-water baseline, and bounded fine-tuning
of the decoder to a user's labelled chips with a portable adapter.

Prithvi-EO-2.0 (Szwarcman et al., 2024) is a ViT-L masked-autoencoder foundation model for Harmonized Landsat
Sentinel-2 imagery; the `TL` variant adds temporal and location embeddings. The checkpoint packaged here is the
upstream authors' fine-tune for flood mapping: the 300 M-parameter encoder, a `LearnedInterpolateToPyramidal` neck
over four encoder depths, a UPerNet decoder and a two-class head, trained on the 446 hand-labelled 512 × 512 chips
of Sen1Floods11 (six bands: blue, green, red, narrow NIR, SWIR 1, SWIR 2) with TerraTorch.

The upstream asset is a PyTorch Lightning checkpoint — a torch zip archive whose pickle references only
`collections.OrderedDict`, `torch._utils._rebuild_tensor_v2` and two storage classes (verified statically by
`audit_pickle`). Under the fleet asset specification (§11) that is executable serialization, so this package
converts it once — `torch.load(weights_only=True)`, the `state_dict` entry, the `model.` prefix stripped — into
safetensors with a pinned digest, and serves only the converted file. The architecture is rebuilt from the
`terratorch` package on PyPI with `backbone_pretrained=False` and loaded strictly; nothing is fetched from the Hub
at load time except the manifest-listed files.

Everything model-related is imported lazily so that snapshot verification, the pickle audit and input validation
run (and can refuse) before `torch` or `terratorch` are imported (fleet RTM-001). `numpy` and `tifffile` are used
for chips and are imported freely.
"""

from __future__ import annotations

import hashlib
import io
import json
import math
import pickletools
import time
import warnings
import zipfile
from collections.abc import Callable, Mapping, Sequence
from dataclasses import dataclass
from pathlib import Path
from typing import Any

MODEL_ID = "ibm-nasa-geospatial/Prithvi-EO-2.0-300M-TL-Sen1Floods11"
MODEL_REVISION = "91ce9d38086a80b078a192b374df758b8855b732"
MODEL_LICENSE = "apache-2.0"
MODEL_KEY = "prithvi-eo-2.0-300m-tl-sen1floods11"
ARTIFACT_FORMAT = "org.valcorza.prithvi-flood-segmentation.adapter.v1"
ARTIFACT_FORMAT_VERSION = "1.0"
ARTIFACT_WEIGHTS_NAME = "adapter.safetensors"
ARTIFACT_MANIFEST_NAME = "manifest.json"
DEFAULT_WEIGHTS_DIR = Path.cwd() / "weights" / MODEL_KEY  # standalone rewrite (build_notebook.py): working-directory-relative
MANIFEST_NAME = "dimer-base-manifest.json"

# Immutable upstream source asset (a Lightning checkpoint, i.e. a pickle; see docs/WEIGHTS.md).
SOURCE_CKPT_NAME = "Prithvi-EO-V2-300M-TL-Sen1Floods11.pt"
SOURCE_CKPT_BYTES = 1_276_843_350
SOURCE_CKPT_SHA256 = "76eed77d8bd543ae441308b80e8408502243a871cb8a338ddc8220ed96dfc270"
# Code-free serving file produced deterministically by `convert_model` (asset spec §11.2).
CONVERTED_WEIGHTS_NAME = "prithvi-eo-2.0-300m-tl-sen1floods11.safetensors"
CONVERTED_SHA256 = "65e4377f96c651dff586bf6ef08c9b8d6c6b59c4dc05e5dfbfe320b264ac47fd"
CONVERTED_BYTES = 1_276_749_320
# Static-audit digest of the source pickle (sorted global names), see `audit_pickle`.
PICKLE_AUDIT_SHA256 = "5b9f0ba08490293d6c17b9cef219991e1a6edda31609429679f8dca1af5a7b10"
CKPT_ALLOWED_GLOBALS = frozenset(
    {"collections.OrderedDict", "torch._utils._rebuild_tensor_v2", "torch.FloatStorage", "torch.LongStorage"}
)
STATE_DICT_PREFIX = "model."

# Architecture (config.yaml of the pinned snapshot) and data-contract facts.
BACKBONE = "prithvi_eo_v2_300_tl"
DECODER = "UperNetDecoder"
DECODER_ARGS: dict[str, Any] = {"decoder_channels": 256}
NECKS: tuple[dict[str, Any], ...] = (
    {"name": "SelectIndices", "indices": [5, 11, 17, 23]},
    {"name": "ReshapeTokensToImage"},
    {"name": "LearnedInterpolateToPyramidal"},
)
HEAD_ARGS: dict[str, Any] = {"head_dropout": 0.1}
PARAMETER_COUNT = 318_968_580  # nn.Parameters; the state dict also carries 208,909 buffer elements
STATE_NUMEL = 319_177_489
STATE_TENSORS = 381
ENCODER_TENSORS = 296
NUM_CLASSES = 2
CLASS_NAMES: tuple[str, ...] = ("no water", "water")
IGNORE_INDEX = -1
BANDS: tuple[str, ...] = ("BLUE", "GREEN", "RED", "NIR_NARROW", "SWIR_1", "SWIR_2")
# Sentinel-2 L1C 13-band order -> the six Prithvi bands (B2, B3, B4, B8A, B11, B12), as upstream's inference script.
S2_L1C_BAND_INDICES: tuple[int, ...] = (1, 2, 3, 8, 11, 12)
# TerraTorch's Sen1Floods11 datamodule statistics for the six bands (reflectance scale, after CONSTANT_SCALE).
MEANS: tuple[float, ...] = (0.1412956, 0.13795798, 0.12353792, 0.30902815, 0.2044958, 0.11912015)
STDS: tuple[float, ...] = (0.07406382, 0.07370365, 0.08692279, 0.11798815, 0.09772074, 0.07659938)
CONSTANT_SCALE = 1e-4  # S2Hand chips are int16 reflectance × 10 000
NO_DATA_VALUES: tuple[float, ...] = (0.0, -9999.0)  # replaced by 0 before normalisation, as upstream
IMAGE_SIZE = 512
MIN_RECORDS = 4
MAX_RECORDS = 2_000
ADAPTATION_MODES = ("decoder", "decoder+last_block")  # the only scopes an adapter may declare
LAST_BLOCK_PREFIX = "encoder.blocks.23."
TRAINABLE_PREFIXES: dict[str, tuple[str, ...]] = {
    "decoder": ("neck.", "decoder.", "head."),
    "decoder+last_block": ("neck.", "decoder.", "head.", LAST_BLOCK_PREFIX),
}


# --------------------------------------------------------------------------------------------------
# manifest, staging, static pickle audit and conversion
# --------------------------------------------------------------------------------------------------


def _sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def _verify_manifest(root: Path, model_id: str, revision: str) -> dict[str, Any]:
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"no snapshot manifest at {manifest_path}")
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    if manifest.get("modelId") != model_id:
        raise ValueError(f"manifest modelId {manifest.get('modelId')!r} != {model_id!r}")
    if manifest.get("revision") != revision:
        raise ValueError(f"manifest revision {manifest.get('revision')!r} != {revision!r}")
    listed = {entry["path"] for entry in manifest["files"]}
    if SOURCE_CKPT_NAME not in listed:
        raise ValueError(f"manifest does not list {SOURCE_CKPT_NAME}; refusing to proceed")
    for entry in manifest["files"]:
        file_path = root / entry["path"]
        if not file_path.is_file():
            raise FileNotFoundError(f"snapshot file missing: {file_path}")
        size = file_path.stat().st_size
        if size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: size {size} != manifest {entry['bytes']}")
        digest = _sha256_file(file_path)
        if digest != entry["sha256"]:
            raise ValueError(f"{entry['path']}: sha256 {digest} != manifest {entry['sha256']}")
        if entry["path"] == SOURCE_CKPT_NAME and (size, digest) != (SOURCE_CKPT_BYTES, SOURCE_CKPT_SHA256):
            raise ValueError(f"{entry['path']}: manifest digest disagrees with the package constant")
    return manifest


def verify_converted(path: str | Path | None = None) -> dict[str, Any]:
    """Check the converted serving file (safetensors) against the pinned digest."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    file_path = root / CONVERTED_WEIGHTS_NAME
    if not file_path.is_file():
        raise FileNotFoundError(f"converted file missing: {file_path}")
    size = file_path.stat().st_size
    if size != CONVERTED_BYTES:
        raise ValueError(f"{CONVERTED_WEIGHTS_NAME}: size {size} != pinned {CONVERTED_BYTES}")
    digest = _sha256_file(file_path)
    if digest != CONVERTED_SHA256:
        raise ValueError(f"{CONVERTED_WEIGHTS_NAME}: sha256 {digest} != pinned {CONVERTED_SHA256}")
    return {"files": [{"path": CONVERTED_WEIGHTS_NAME, "bytes": size, "sha256": digest}]}


def verify_snapshot(path: str | Path | None = None) -> dict[str, Any]:
    """Check the snapshot against its DIMER manifest (size + SHA-256 of every listed Hub file) and, when the
    converted serving file is present, that against the pinned digest."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest = _verify_manifest(root, MODEL_ID, MODEL_REVISION)
    converted = (root / CONVERTED_WEIGHTS_NAME).is_file()
    if converted:
        verify_converted(root)
    return {**manifest, "converted": converted}


def _hub_download(relative_path: str, root: Path) -> None:
    """Fetch one manifest-listed file at the pinned revision straight into the snapshot directory."""
    from huggingface_hub import hf_hub_download

    hf_hub_download(MODEL_ID, relative_path, revision=MODEL_REVISION, local_dir=str(root))


def stage_missing_files(
    path: str | Path | None = None,
    *,
    allow_download: bool = False,
    downloader: Callable[[str, Path], None] | None = None,
) -> list[str]:
    """Fetch manifest entries that are absent locally (a fresh clone commits the manifest and git-ignores the
    1.28 GB checkpoint and the safetensors it converts to)."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID or manifest.get("revision") != MODEL_REVISION:
        raise ValueError(
            f"manifest names {manifest.get('modelId')}@{manifest.get('revision')}, "
            f"package pins {MODEL_ID}@{MODEL_REVISION}; refusing to stage"
        )
    missing = [entry["path"] for entry in manifest["files"] if not (root / entry["path"]).is_file()]
    if not missing:
        return []
    if not allow_download:
        raise FileNotFoundError(
            f"snapshot at {root} is missing {missing}; pass allow_download=True to fetch them at {MODEL_REVISION}"
        )
    fetch = downloader or _hub_download
    for relative_path in missing:
        fetch(relative_path, root)
    return missing


def _pickle_globals(data: bytes) -> dict[str, int]:
    """Every global a pickle stream would import, collected with `pickletools.genops` (no execution)."""
    found: dict[str, int] = {}
    stack: list[Any] = []
    for op, arg, _pos in pickletools.genops(io.BytesIO(data)):
        if op.name == "GLOBAL":  # pickletools renders the (module, name) pair space-separated
            key = arg.replace("\n", " ").replace(" ", ".", 1)
            found[key] = found.get(key, 0) + 1
        elif op.name == "STACK_GLOBAL":
            key = f"{stack[-2]}.{stack[-1]}"
            found[key] = found.get(key, 0) + 1
        if op.name in ("SHORT_BINUNICODE", "BINUNICODE", "UNICODE", "SHORT_BINSTRING", "BINSTRING"):
            stack.append(arg)
        elif op.name in ("MEMOIZE", "BINPUT", "LONG_BINPUT", "PUT"):
            pass
        else:
            stack.append(None)
    return found


def audit_pickle(path: str | Path, *, allowed: frozenset[str] = CKPT_ALLOWED_GLOBALS) -> dict[str, Any]:
    """Statically list the globals a pickle (plain, or inside a torch zip archive) would import and refuse any
    outside `allowed`. Executes nothing. Returns the sorted globals and their digest."""
    file_path = Path(path)
    if not file_path.is_file():
        raise FileNotFoundError(f"file not found: {file_path}")
    data = file_path.read_bytes()
    found: dict[str, int] = {}
    nested = 0
    if data[:4] == b"PK\x03\x04":
        archive = zipfile.ZipFile(io.BytesIO(data))
        for name in archive.namelist():
            if name.endswith(".pkl"):
                nested += 1
                for key, count in _pickle_globals(archive.read(name)).items():
                    found[key] = found.get(key, 0) + count
    else:
        found = _pickle_globals(data)
    violations = sorted(name for name in found if name not in allowed)
    summary = {
        "file": file_path.name,
        "torch_archive": data[:4] == b"PK\x03\x04",
        "pickles": nested if nested else 1,
        "globals": sorted(found),
        "violations": violations,
        "audit_sha256": hashlib.sha256("\n".join(sorted(found)).encode("utf-8")).hexdigest(),
    }
    if violations:
        raise ValueError(f"{file_path.name}: pickle audit failed, globals outside the allow-list: {violations}")
    return summary


def _check_pinned_source(root: Path) -> dict[str, Any]:
    source = root / SOURCE_CKPT_NAME
    if not source.is_file():
        raise FileNotFoundError(f"source file not found: {source}")
    size = source.stat().st_size
    if size != SOURCE_CKPT_BYTES:
        raise ValueError(f"{SOURCE_CKPT_NAME}: size {size} != pinned {SOURCE_CKPT_BYTES}")
    digest = _sha256_file(source)
    if digest != SOURCE_CKPT_SHA256:
        raise ValueError(f"{SOURCE_CKPT_NAME}: sha256 {digest} != pinned {SOURCE_CKPT_SHA256}")
    audit = audit_pickle(source)
    if audit["audit_sha256"] != PICKLE_AUDIT_SHA256:
        raise ValueError(f"{SOURCE_CKPT_NAME}: pickle audit digest {audit['audit_sha256']} != pinned {PICKLE_AUDIT_SHA256}")
    return {"path": SOURCE_CKPT_NAME, "bytes": size, "sha256": digest, "audit": audit}


def build_model() -> Any:
    """Instantiate the fine-tuned architecture from the installed `terratorch` package (no pretrained download)."""
    from terratorch.models import EncoderDecoderFactory

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        return EncoderDecoderFactory().build_model(
            task="segmentation",
            backbone=BACKBONE,
            backbone_pretrained=False,
            backbone_bands=list(BANDS),
            decoder=DECODER,
            num_classes=NUM_CLASSES,
            rescale=True,
            necks=[dict(n) for n in NECKS],
            **DECODER_ARGS,
            **HEAD_ARGS,
        )


def convert_model(path: str | Path | None = None) -> dict[str, Any]:
    """Convert the pinned Lightning checkpoint into safetensors, deterministically, after size, digest and
    static-audit checks: torch's weights-only unpickler, the `state_dict` entry with the `model.` prefix
    stripped, a strict load into the rebuilt architecture, and the model's own state dict saved."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    source = _check_pinned_source(root)
    import torch
    from safetensors.torch import save_file

    started = time.perf_counter()
    payload = torch.load(root / SOURCE_CKPT_NAME, map_location="cpu", weights_only=True)
    if not isinstance(payload, dict) or "state_dict" not in payload:
        raise ValueError(f"{SOURCE_CKPT_NAME} did not unpickle to a Lightning checkpoint with a state_dict")
    state = payload["state_dict"]
    if not isinstance(state, dict) or any(not isinstance(v, torch.Tensor) for v in state.values()):
        raise ValueError(f"{SOURCE_CKPT_NAME}: state_dict is not a dict of tensors")
    stripped = {}
    for key, value in state.items():
        if not key.startswith(STATE_DICT_PREFIX):
            raise ValueError(f"{SOURCE_CKPT_NAME}: unexpected state-dict key {key!r} outside {STATE_DICT_PREFIX!r}")
        stripped[key[len(STATE_DICT_PREFIX) :]] = value
    model = build_model()
    model.load_state_dict(stripped, strict=True)
    canonical = {k: v.contiguous() for k, v in model.state_dict().items()}
    n_params = sum(v.numel() for v in canonical.values())
    if len(canonical) != STATE_TENSORS or n_params != STATE_NUMEL:
        raise ValueError(
            f"converted state dict has {len(canonical)} tensors / {n_params} elements; expected {STATE_TENSORS} / {STATE_NUMEL}"
        )
    save_file(canonical, str(root / CONVERTED_WEIGHTS_NAME), metadata={"format": "pt"})
    report = verify_converted(root)
    return {
        "source": {k: v for k, v in source.items() if k != "audit"},
        "audit": source["audit"],
        "checkpoint": {
            "epoch": payload.get("epoch"),
            "global_step": payload.get("global_step"),
            "lightning_version": payload.get("pytorch-lightning_version"),
        },
        "converted": report["files"],
        "seconds": round(time.perf_counter() - started, 2),
    }


# --------------------------------------------------------------------------------------------------
# chips, labels and validation (no model import)
# --------------------------------------------------------------------------------------------------

INPUT_SCHEMA: dict[str, Any] = {
    "record": (
        "{id, image, label?}: image = (6, 512, 512) float32 reflectance chip (or a GeoTIFF path); "
        "label = (512, 512) int mask with 0/1/-1 (or a GeoTIFF path), optional"
    ),
    "bands": list(BANDS),
    "image_size": IMAGE_SIZE,
    "value_units": (
        "surface/TOA reflectance × 10 000 (int16, the Sen1Floods11 S2Hand encoding) or reflectance in [0, 1]; "
        "values above 1 are scaled by 1e-4"
    ),
    "no_data": list(NO_DATA_VALUES),
    "classes": {str(i): name for i, name in enumerate(CLASS_NAMES)},
    "ignore_index": IGNORE_INDEX,
    "records": [MIN_RECORDS, MAX_RECORDS],
    "validation": (
        "record shape, band count, chip size, finiteness, value range and label values only. Nothing checks that "
        "the bands are the six Prithvi bands in the right order, that the reflectance is atmospherically corrected, "
        "or that the label was drawn for this chip -- any six-band 512 × 512 array is segmented without complaint"
    ),
}


def _read_tiff(path: Path) -> Any:
    """Read a GeoTIFF's pixel array with tifffile as (bands, H, W) or (H, W); no georeferencing is used."""
    import numpy as np
    import tifffile

    with tifffile.TiffFile(path) as tf:
        array = tf.asarray()
        planar = tf.pages[0].planarconfig
    if array.ndim == 3 and planar is not None and int(planar) == 1 and array.shape[-1] <= 16:
        array = np.moveaxis(array, -1, 0)  # pixel-interleaved -> band-sequential
    return np.asarray(array)


def read_chip(path: str | Path, *, band_indices: Sequence[int] | None = None) -> Any:
    """Load a chip from a GeoTIFF as float32 (6, H, W); `band_indices` selects the six Prithvi bands from a
    wider stack (e.g. S2_L1C_BAND_INDICES for a 13-band Sentinel-2 L1C file)."""
    import numpy as np

    array = _read_tiff(Path(path))
    if array.ndim != 3:
        raise ValueError(f"{Path(path).name}: expected a multi-band raster, got shape {array.shape}")
    if band_indices is not None:
        array = array[list(band_indices)]
    elif array.shape[0] != len(BANDS) and array.shape[0] == 13:
        array = array[list(S2_L1C_BAND_INDICES)]
    return np.ascontiguousarray(array.astype(np.float32))


def read_mask(path: str | Path) -> Any:
    """Load a label raster from a GeoTIFF as int64 (H, W)."""
    import numpy as np

    array = _read_tiff(Path(path))
    if array.ndim == 3:
        if array.shape[0] != 1:
            raise ValueError(f"{Path(path).name}: a label raster must have one band, got shape {array.shape}")
        array = array[0]
    return np.ascontiguousarray(array.astype(np.int64))


def _check_record(record: Any, index: int) -> dict[str, Any]:
    import numpy as np

    label_name = f"records[{index}]"
    if not isinstance(record, Mapping):
        raise ValueError(f"{label_name} must be a mapping with id/image[/label]")
    for key in ("id", "image"):
        if key not in record:
            raise ValueError(f"{label_name} is missing {key!r}")
    rid, image = record["id"], record["image"]
    if not isinstance(rid, str) or not rid or len(rid) > 128:
        raise ValueError(f"{label_name}: id must be a non-empty string of at most 128 characters")
    if isinstance(image, str | Path):
        if not Path(image).is_file():
            raise ValueError(f"{label_name}: image file not found: {image}")
        image = read_chip(image)
    try:
        array = np.asarray(image, dtype=np.float32)
    except (TypeError, ValueError) as exc:
        raise ValueError(f"{label_name}: image must be a numeric array") from exc
    if array.shape != (len(BANDS), IMAGE_SIZE, IMAGE_SIZE):
        raise ValueError(f"{label_name}: image must have shape {(len(BANDS), IMAGE_SIZE, IMAGE_SIZE)}, got {array.shape}")
    if not np.all(np.isfinite(array)):
        raise ValueError(f"{label_name}: image contains non-finite values")
    for value in NO_DATA_VALUES:
        array = np.where(array == value, 0.0, array)
    if float(array.max()) > 1.0:
        array = array * CONSTANT_SCALE
    if float(array.min()) < -0.5 or float(array.max()) > 2.0:
        span = (float(array.min()), float(array.max()))
        raise ValueError(f"{label_name}: reflectance outside the plausible range after scaling: {span}")
    item: dict[str, Any] = {"id": rid, "image": np.ascontiguousarray(array.astype(np.float32))}
    label = record.get("label")
    if label is not None:
        if isinstance(label, str | Path):
            if not Path(label).is_file():
                raise ValueError(f"{label_name}: label file not found: {label}")
            label = read_mask(label)
        try:
            mask = np.asarray(label)
        except (TypeError, ValueError) as exc:
            raise ValueError(f"{label_name}: label must be an integer array") from exc
        if mask.shape != (IMAGE_SIZE, IMAGE_SIZE):
            raise ValueError(f"{label_name}: label must have shape {(IMAGE_SIZE, IMAGE_SIZE)}, got {mask.shape}")
        if not np.issubdtype(mask.dtype, np.integer) and not np.all(mask == np.round(mask)):
            raise ValueError(f"{label_name}: label values must be integers")
        allowed = set(range(NUM_CLASSES)) | {IGNORE_INDEX}
        found = set(np.unique(mask).astype(int).tolist())
        if not found <= allowed:
            raise ValueError(f"{label_name}: label values {sorted(found - allowed)} outside {sorted(allowed)}")
        item["label"] = np.ascontiguousarray(mask.astype(np.int64))
    for key in ("split", "region", "source", "source_id"):
        if key in record:
            item[key] = record[key]
    return item


def check_record(record: Mapping[str, Any]) -> dict[str, Any]:
    """Validate one record and return its normalised copy (float32 reflectance, no-data replaced, int64 label)."""
    return _check_record(record, 0)


def chip_digest(record: Mapping[str, Any]) -> str:
    checked = _check_record(record, 0)
    digest = hashlib.sha256(checked["image"].tobytes())
    if "label" in checked:
        digest.update(checked["label"].tobytes())
    return digest.hexdigest()


def dataset_digest(records: Sequence[Mapping[str, Any]]) -> str:
    payload = [[r["id"], chip_digest(r)] for r in records]
    return hashlib.sha256(json.dumps(payload, separators=(",", ":")).encode("utf-8")).hexdigest()


def validate_dataset(
    records: Sequence[Mapping[str, Any]],
    *,
    min_records: int = MIN_RECORDS,
    max_records: int = MAX_RECORDS,
    require_labels: bool = True,
) -> dict[str, Any]:
    """Structural validation of a chip dataset; raises ValueError before any model import."""
    import numpy as np

    if isinstance(records, Mapping) or not isinstance(records, Sequence) or isinstance(records, str | bytes):
        raise ValueError("records must be a list of {id, image, label} mappings")
    if not min_records <= len(records) <= max_records:
        raise ValueError(f"{len(records)} records; {min_records}..{max_records} are required")
    checked = []
    ids: set[str] = set()
    class_pixels = np.zeros(NUM_CLASSES, dtype=np.int64)
    ignored = 0
    for index, record in enumerate(records):
        item = _check_record(record, index)
        if item["id"] in ids:
            raise ValueError(f"duplicate id {item['id']!r}")
        ids.add(item["id"])
        if require_labels and "label" not in item:
            raise ValueError(f"records[{index}] has no label; every record of a labelled dataset needs one")
        if "label" in item:
            for c in range(NUM_CLASSES):
                class_pixels[c] += int((item["label"] == c).sum())
            ignored += int((item["label"] == IGNORE_INDEX).sum())
        checked.append(item)
    labelled = sum("label" in r for r in checked)
    if require_labels and labelled and class_pixels[1] == 0:
        raise ValueError(f"no pixel of class 1 ({CLASS_NAMES[1]}) in the dataset; nothing to learn or evaluate")
    total = int(class_pixels.sum())
    return {
        "records": checked,
        "n_records": len(checked),
        "n_labelled": labelled,
        "image_size": IMAGE_SIZE,
        "bands": list(BANDS),
        "class_pixel_fraction": {
            CLASS_NAMES[c]: round(float(class_pixels[c]) / total, 4) if total else None for c in range(NUM_CLASSES)
        },
        "ignored_pixels": ignored,
        "reflectance_range": [
            round(float(min(r["image"].min() for r in checked)), 4),
            round(float(max(r["image"].max() for r in checked)), 4),
        ],
        "digest": dataset_digest(checked),
        "model_id": MODEL_ID,
    }


def validate_inputs(record: Mapping[str, Any]) -> dict[str, Any]:
    """Validate one record; returns its id, shape, reflectance range and label class fractions."""
    item = _check_record(record, 0)
    report = {
        "id": item["id"],
        "shape": tuple(item["image"].shape),
        "reflectance_range": [round(float(item["image"].min()), 4), round(float(item["image"].max()), 4)],
        "has_label": "label" in item,
    }
    if "label" in item:
        label = item["label"]
        valid = int((label != IGNORE_INDEX).sum())
        report["label_fraction"] = {
            CLASS_NAMES[c]: round(float((label == c).sum()) / max(valid, 1), 4) for c in range(NUM_CLASSES)
        }
        report["ignored_pixels"] = int((label == IGNORE_INDEX).sum())
    return report


# --------------------------------------------------------------------------------------------------
# pipeline
# --------------------------------------------------------------------------------------------------


def _normalise(images: Any) -> Any:
    """(B, 6, H, W) reflectance -> standardised with the upstream datamodule statistics."""
    import numpy as np

    mean = np.asarray(MEANS, dtype=np.float32)[None, :, None, None]
    std = np.asarray(STDS, dtype=np.float32)[None, :, None, None]
    return (images - mean) / std


@dataclass
class PrithviFloodPipeline:
    """Flood-extent segmentation and bounded decoder fine-tuning on top of the verified Prithvi flood model."""

    model: Any
    device: str
    weights_dir: Path
    source: str
    adapter: dict[str, Any] | None = None

    @classmethod
    def from_pretrained(
        cls,
        *,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
        require_source: bool = True,
        report: Callable[[dict[str, Any]], None] | None = None,
    ) -> PrithviFloodPipeline:
        """Verify, convert if needed, rebuild from the installed package and strictly load. With
        `require_source=False` the checkpoint may be absent (the DIMER-hosted case) as long as the converted file
        verifies. `report` receives the audit and conversion records when a conversion happens."""
        root = Path(weights_dir) if weights_dir is not None else DEFAULT_WEIGHTS_DIR
        if require_source:
            stage_missing_files(root, allow_download=allow_download)
            snapshot = verify_snapshot(root)
            if not snapshot["converted"]:
                conversion = convert_model(root)
                if report is not None:
                    report({"conversion": conversion})
                snapshot = verify_snapshot(root)
            elif report is not None:
                report({"conversion": "converted file already present and digest-verified"})
            source = "converted from the manifest-verified source checkpoint"
        else:
            verify_converted(root)
            source = "converted file, pinned digest (source checkpoint not required)"
        import torch
        from safetensors.torch import load_file

        chosen = device or ("cuda" if torch.cuda.is_available() else "cpu")
        if chosen.startswith("cuda") and not torch.cuda.is_available():
            raise ValueError("device='cuda' requested but CUDA is not available")
        model = build_model()
        state = load_file(str(root / CONVERTED_WEIGHTS_NAME))
        model.load_state_dict(state, strict=True)
        n_params = sum(p.numel() for p in model.parameters())
        if n_params != PARAMETER_COUNT:
            raise ValueError(f"rebuilt model has {n_params} parameters, expected {PARAMETER_COUNT}")
        model.to(torch.device(chosen)).eval()
        for param in model.parameters():
            param.requires_grad_(False)
        return cls(model=model, device=chosen, weights_dir=root, source=source)

    # ---- forward ---------------------------------------------------------------------------------------

    def _logits(self, images: Any, *, grad: bool = False) -> Any:
        """(B, 6, H, W) float32 reflectance -> (B, 2, H, W) float32 logits at input resolution."""
        import numpy as np
        import torch

        batch = torch.from_numpy(_normalise(np.asarray(images, dtype=np.float32))).to(self.device)
        use_amp = self.device.startswith("cuda")
        context = torch.enable_grad() if grad else torch.inference_mode()
        with context, torch.autocast(device_type=self.device.split(":")[0], dtype=torch.float16, enabled=use_amp):
            out = self.model(batch)
        logits = out.output if hasattr(out, "output") else out
        if tuple(logits.shape[-2:]) != tuple(batch.shape[-2:]):
            logits = torch.nn.functional.interpolate(logits.float(), size=batch.shape[-2:], mode="bilinear", align_corners=False)
        return logits.float()

    # ---- inference -------------------------------------------------------------------------------------

    def predict(self, records: Sequence[Mapping[str, Any]], *, batch_size: int = 4) -> dict[str, Any]:
        """Segment chips: per record the argmax mask (H, W) uint8, the softmax scores (2, H, W) float32 and the
        fraction of pixels in each class. Softmax scores are the model's own outputs, not calibrated probabilities."""
        import numpy as np
        import torch

        checked = validate_dataset(records, min_records=1, require_labels=False)["records"]
        if not isinstance(batch_size, int) or not 1 <= batch_size <= 32:
            raise ValueError("batch_size must be an int in 1..32")
        started = time.perf_counter()
        predictions = []
        for start in range(0, len(checked), batch_size):
            batch = checked[start : start + batch_size]
            logits = self._logits(np.stack([r["image"] for r in batch]))
            scores = torch.softmax(logits, dim=1).cpu().numpy()
            masks = scores.argmax(axis=1).astype(np.uint8)
            for record, score, mask in zip(batch, scores, masks, strict=True):
                predictions.append(
                    {
                        "id": record["id"],
                        "mask": mask,
                        "scores": score.astype(np.float32),
                        "class_fraction": {CLASS_NAMES[c]: round(float((mask == c).mean()), 4) for c in range(NUM_CLASSES)},
                    }
                )
        return {
            "model": {"id": MODEL_ID, "revision": MODEL_REVISION, "key": MODEL_KEY, "adapted": self.adapter is not None},
            "classes": list(CLASS_NAMES),
            "decision_rule": "argmax over the two class scores (no threshold)",
            "predictions": predictions,
            "seconds": round(time.perf_counter() - started, 3),
        }

    def evaluate(self, records: Sequence[Mapping[str, Any]], *, batch_size: int = 4) -> dict[str, Any]:
        """Pixel-level metrics on labelled chips (ignore index excluded): per-class IoU, mean IoU, accuracy and the
        class-1 F1, precision and recall, with the all-class-0 baseline scored on the same pixels."""
        pass  # standalone rewrite (build_notebook.py): `from .metrics import majority_baseline, segmentation_metrics` removed — names are kernel globals defined by the carried modules

        checked = validate_dataset(records, min_records=1)["records"]
        started = time.perf_counter()
        result = self.predict(checked, batch_size=batch_size)
        masks = [p["mask"] for p in result["predictions"]]
        labels = [r["label"] for r in checked]
        metrics = segmentation_metrics(masks, labels)
        return {
            "n_records": len(checked),
            "metric": "pixel IoU / F1 over the labelled pixels of the held-out chips (ignore index excluded)",
            "model": metrics,
            "baseline_no_water": majority_baseline(labels),
            "adapted": self.adapter is not None,
            "seconds": round(time.perf_counter() - started, 3),
        }

    # ---- adaptation ------------------------------------------------------------------------------------

    def _trainable(self, mode: str) -> list[str]:
        if mode not in ADAPTATION_MODES:
            raise ValueError(f"trainable must be one of {ADAPTATION_MODES}")
        prefixes = TRAINABLE_PREFIXES[mode]
        return sorted(name for name, _param in self.model.named_parameters() if name.startswith(prefixes))

    def adapt(
        self,
        train: Sequence[Mapping[str, Any]],
        val: Sequence[Mapping[str, Any]] | None = None,
        *,
        epochs: int = 4,
        lr: float = 1e-5,
        batch_size: int = 2,
        trainable: str = "decoder",
        seed: int = 0,
        progress: Callable[[dict[str, Any]], None] | None = None,
    ) -> dict[str, Any]:
        """Bounded fine-tuning of the neck, decoder and head (`trainable="decoder"`; `"decoder+last_block"` also
        unfreezes the last encoder block) on labelled chips: cross-entropy over the labelled pixels (ignore index
        excluded), AdamW at a fixed learning rate, seeded horizontal/vertical flips, float16 autocast with loss
        scaling on CUDA. Epoch 0 records the frozen model; the epoch with the lowest validation loss is kept."""
        if not isinstance(epochs, int) or not 1 <= epochs <= 50:
            raise ValueError("epochs must be an int in 1..50")
        if not (0.0 < lr <= 1e-2):
            raise ValueError("lr must be in (0, 1e-2]")
        if not isinstance(batch_size, int) or not 1 <= batch_size <= 16:
            raise ValueError("batch_size must be an int in 1..16")
        names = self._trainable(trainable)
        train_checked = validate_dataset(train)["records"]
        val_checked = validate_dataset(val, min_records=1)["records"] if val is not None else None
        import numpy as np
        import torch

        torch.manual_seed(seed)
        started = time.perf_counter()
        model = self.model
        name_set = set(names)
        for name, param in model.named_parameters():
            param.requires_grad_(name in name_set)
        params = [p for n, p in model.named_parameters() if n in name_set]
        n_trainable = sum(p.numel() for p in params)
        optimiser = torch.optim.AdamW(params, lr=lr, weight_decay=0.0)
        use_amp = self.device.startswith("cuda")
        scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
        rng = np.random.default_rng(seed)

        def val_loss() -> float | None:
            if val_checked is None:
                return None
            model.eval()
            losses = []
            for start in range(0, len(val_checked), batch_size):
                batch = val_checked[start : start + batch_size]
                logits = self._logits(np.stack([r["image"] for r in batch]))
                target = torch.from_numpy(np.stack([r["label"] for r in batch])).to(self.device)
                losses.append(float(torch.nn.functional.cross_entropy(logits, target, ignore_index=IGNORE_INDEX)))
            return sum(losses) / len(losses)

        initial_state = {k: v.detach().clone() for k, v in model.state_dict().items() if k in name_set}
        try:
            history: list[dict[str, Any]] = []
            entry: dict[str, Any] = {"epoch": 0, "train_loss": None, "val_loss": val_loss(), "note": "frozen model"}
            if val_checked is not None:
                entry["val"] = self.evaluate(val_checked, batch_size=batch_size)["model"]
            history.append(entry)
            best_val = entry["val_loss"] if entry["val_loss"] is not None else math.inf
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items() if k in name_set}
            best_epoch = 0
            if progress:
                progress(entry)
            n_steps = 0
            for epoch in range(1, epochs + 1):
                model.train()
                for module in model.modules():  # BatchNorm statistics stay frozen: tiny batches would corrupt them
                    if isinstance(module, torch.nn.modules.batchnorm._BatchNorm):
                        module.eval()
                order = rng.permutation(len(train_checked)).tolist()
                losses = []
                for start in range(0, len(order), batch_size):
                    batch = [train_checked[i] for i in order[start : start + batch_size]]
                    images = np.stack([r["image"] for r in batch])
                    labels = np.stack([r["label"] for r in batch])
                    if rng.random() < 0.5:
                        images, labels = images[..., ::-1], labels[..., ::-1]
                    if rng.random() < 0.5:
                        images, labels = images[..., ::-1, :], labels[..., ::-1, :]
                    logits = self._logits(np.ascontiguousarray(images), grad=True)
                    target = torch.from_numpy(np.ascontiguousarray(labels)).to(self.device)
                    loss = torch.nn.functional.cross_entropy(logits, target, ignore_index=IGNORE_INDEX)
                    optimiser.zero_grad(set_to_none=True)
                    scaler.scale(loss).backward()
                    scaler.unscale_(optimiser)
                    torch.nn.utils.clip_grad_norm_(params, 1.0)
                    scaler.step(optimiser)
                    scaler.update()
                    losses.append(float(loss.detach()))
                    n_steps += 1
                model.eval()
                entry = {"epoch": epoch, "train_loss": sum(losses) / len(losses), "val_loss": val_loss()}
                if val_checked is not None:
                    entry["val"] = self.evaluate(val_checked, batch_size=batch_size)["model"]
                history.append(entry)
                if progress:
                    progress(entry)
                if entry["val_loss"] is None or entry["val_loss"] < best_val:
                    best_val = entry["val_loss"] if entry["val_loss"] is not None else best_val
                    best_state = {k: v.detach().clone() for k, v in model.state_dict().items() if k in name_set}
                    best_epoch = epoch
        except BaseException:
            # Transactional: a failure in training, validation or the progress callback leaves the model as it
            # was before adapt() (trained tensors restored), frozen, with no adapter attached.
            restore = dict(model.state_dict())
            restore.update(initial_state)
            model.load_state_dict(restore, strict=True)
            model.eval()
            for param in model.parameters():
                param.requires_grad_(False)
            self.adapter = None
            raise
        merged = dict(model.state_dict())
        merged.update(best_state)
        model.load_state_dict(merged, strict=True)
        model.eval()
        for param in model.parameters():
            param.requires_grad_(False)
        self.adapter = {
            "trainable": trainable,
            "trainable_names": names,
            "n_trainable": n_trainable,
            "n_total": sum(p.numel() for p in model.parameters()),
            "epochs": epochs,
            "best_epoch": best_epoch,
            "lr": lr,
            "batch_size": batch_size,
            "augmentation": "seeded horizontal/vertical flips",
            "batchnorm": "running statistics frozen (eval mode) during adaptation",
            "precision": "float16 autocast + GradScaler" if use_amp else "float32",
            "n_train_records": len(train_checked),
            "n_steps": n_steps,
            "seed": seed,
            "history": history,
            "seconds": round(time.perf_counter() - started, 2),
        }
        return dict(self.adapter)

    # ---- artifacts -------------------------------------------------------------------------------------

    def save_artifact(self, output_dir: str | Path, metadata: Mapping[str, Any] | None = None) -> Path:
        """Write the adapted tensors as safetensors with a manifest."""
        if self.adapter is None:
            raise ValueError("nothing to save: call adapt() first")
        from safetensors.torch import save_file

        out = Path(output_dir)
        out.mkdir(parents=True, exist_ok=True)
        names = set(self.adapter["trainable_names"])
        tensors = {k: v.detach().cpu().contiguous() for k, v in self.model.state_dict().items() if k in names}
        weights_path = out / ARTIFACT_WEIGHTS_NAME
        save_file(tensors, str(weights_path), metadata={"format": "pt"})
        manifest = {
            "format": ARTIFACT_FORMAT,
            "format_version": ARTIFACT_FORMAT_VERSION,
            "base_model": {"id": MODEL_ID, "revision": MODEL_REVISION, "key": MODEL_KEY, "converted_sha256": CONVERTED_SHA256},
            "adapter": {k: v for k, v in self.adapter.items() if k not in ("history", "trainable_names")},
            "history": self.adapter["history"],
            "tensors": sorted(tensors),
            "files": [
                {"path": ARTIFACT_WEIGHTS_NAME, "bytes": weights_path.stat().st_size, "sha256": _sha256_file(weights_path)}
            ],
            "metadata": dict(metadata or {}),
        }
        (out / ARTIFACT_MANIFEST_NAME).write_text(json.dumps(manifest, indent=2), encoding="utf-8")
        return out

    @staticmethod
    def check_artifact_manifest(root: Path, manifest: Mapping[str, Any]) -> tuple[Path, str]:
        """Static checks on an adapter manifest, before any model or weights work: format and version, the pinned
        base and converted digest, exactly one weights entry named `adapter.safetensors` inside the artifact
        directory, and an adaptation mode that is one of the declared scopes. Returns the weights path and mode."""
        if manifest.get("format") != ARTIFACT_FORMAT:
            raise ValueError(f"artifact format {manifest.get('format')!r} != {ARTIFACT_FORMAT!r}")
        if manifest.get("format_version") != ARTIFACT_FORMAT_VERSION:
            raise ValueError(
                f"artifact format_version {manifest.get('format_version')!r} is not supported "
                f"(expected {ARTIFACT_FORMAT_VERSION!r})"
            )
        base = manifest.get("base_model", {})
        if (base.get("id"), base.get("revision")) != (MODEL_ID, MODEL_REVISION):
            raise ValueError("artifact was adapted from a different base model or revision")
        if base.get("converted_sha256") != CONVERTED_SHA256:
            raise ValueError("artifact records a different converted-base digest")
        files = manifest.get("files")
        if not isinstance(files, list) or len(files) != 1:
            raise ValueError("artifact manifest must list exactly one weights file")
        entry = files[0]
        if not isinstance(entry, Mapping) or entry.get("path") != ARTIFACT_WEIGHTS_NAME:
            raise ValueError(f"artifact weights file must be named {ARTIFACT_WEIGHTS_NAME!r}")
        weights_path = (root / entry["path"]).resolve()
        if weights_path.parent != root.resolve():
            raise ValueError("artifact weights file must sit inside the artifact directory")
        adapter = manifest.get("adapter")
        mode = adapter.get("trainable") if isinstance(adapter, Mapping) else None
        if mode not in ADAPTATION_MODES:
            raise ValueError(f"artifact adapter.trainable must be one of {ADAPTATION_MODES}")
        if not isinstance(manifest.get("tensors"), list):
            raise ValueError("artifact manifest must list its tensors")
        return weights_path, mode

    def load_artifact(self, artifact_dir: str | Path) -> dict[str, Any]:
        """Verify an adapter's manifest, scope and digest, then overwrite exactly the tensors the scope allows."""
        root = Path(artifact_dir)
        manifest = json.loads((root / ARTIFACT_MANIFEST_NAME).read_text(encoding="utf-8"))
        weights_path, mode = self.check_artifact_manifest(root, manifest)
        expected = self._trainable(mode)
        if sorted(manifest["tensors"]) != expected:
            raise ValueError(
                f"artifact tensor list does not match the {len(expected)} tensors that trainable={mode!r} may change"
            )
        entry = manifest["files"][0]
        if _sha256_file(weights_path) != entry["sha256"] or weights_path.stat().st_size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: digest or size mismatch; refusing to load")
        from safetensors.torch import load_file

        tensors = load_file(str(weights_path))
        if sorted(tensors) != expected:
            raise ValueError("artifact tensor names differ from the validated manifest")
        state = self.model.state_dict()
        for key, value in tensors.items():
            if tuple(value.shape) != tuple(state[key].shape):
                raise ValueError(f"artifact tensor {key} has shape {tuple(value.shape)}, model has {tuple(state[key].shape)}")
        merged = dict(state)
        merged.update({k: v.to(state[k].device, state[k].dtype) for k, v in tensors.items()})
        self.model.load_state_dict(merged, strict=True)
        self.model.eval()
        self.adapter = {**manifest["adapter"], "trainable_names": manifest["tensors"], "history": manifest.get("history", [])}
        return manifest

    @classmethod
    def from_artifact(
        cls,
        artifact_dir: str | Path,
        *,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
        require_source: bool = True,
    ) -> PrithviFloodPipeline:
        root = Path(artifact_dir)
        manifest = json.loads((root / ARTIFACT_MANIFEST_NAME).read_text(encoding="utf-8"))
        cls.check_artifact_manifest(root, manifest)
        pipeline = cls.from_pretrained(
            device=device, weights_dir=weights_dir, allow_download=allow_download, require_source=require_source
        )
        pipeline.load_artifact(artifact_dir)
        return pipeline

**Module 2/3:** `src/prithvi_flood_segmentation_pipeline/metrics.py` (carried verbatim; see the note above)

In [ ]:
"""Pixel-level segmentation metrics from a confusion matrix over the labelled pixels (ignore index excluded):
per-class IoU, mean IoU, overall accuracy, and the positive class's precision, recall and F1 — plus the
"no water" baseline that predicts class 0 everywhere, scored on exactly the same pixels.
"""

from __future__ import annotations

from collections.abc import Sequence
from typing import Any

# standalone rewrite (build_notebook.py): `from .pipeline import CLASS_NAMES, IGNORE_INDEX, NUM_CLASSES` removed — names are kernel globals defined by the carried modules


def confusion_matrix(predictions: Sequence[Any], labels: Sequence[Any]) -> Any:
    """(NUM_CLASSES, NUM_CLASSES) counts, rows = truth, columns = prediction; ignore-index pixels are skipped."""
    import numpy as np

    if len(predictions) != len(labels) or not labels:
        raise ValueError("predictions and labels must be non-empty sequences of equal length")
    matrix = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=np.int64)
    for pred, label in zip(predictions, labels, strict=True):
        pred = np.asarray(pred).astype(np.int64)
        label = np.asarray(label).astype(np.int64)
        if pred.shape != label.shape:
            raise ValueError(f"prediction shape {pred.shape} != label shape {label.shape}")
        keep = label != IGNORE_INDEX
        if not np.all((pred[keep] >= 0) & (pred[keep] < NUM_CLASSES)) or not np.all(label[keep] < NUM_CLASSES):
            raise ValueError("class ids outside 0..NUM_CLASSES-1")
        matrix += np.bincount(label[keep] * NUM_CLASSES + pred[keep], minlength=NUM_CLASSES**2).reshape(NUM_CLASSES, NUM_CLASSES)
    return matrix


def metrics_from_confusion(matrix: Any) -> dict[str, Any]:
    import numpy as np

    matrix = np.asarray(matrix, dtype=np.float64)
    total = matrix.sum()
    if total == 0:
        raise ValueError("no labelled pixel to score")
    tp = np.diag(matrix)
    fp = matrix.sum(axis=0) - tp
    fn = matrix.sum(axis=1) - tp
    union = tp + fp + fn
    iou = np.where(union > 0, tp / np.maximum(union, 1), np.nan)
    present = matrix.sum(axis=1) > 0
    pos = NUM_CLASSES - 1
    precision = tp[pos] / (tp[pos] + fp[pos]) if tp[pos] + fp[pos] > 0 else 0.0
    recall = tp[pos] / (tp[pos] + fn[pos]) if tp[pos] + fn[pos] > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall > 0 else 0.0
    return {
        "iou": {CLASS_NAMES[c]: (round(float(iou[c]), 4) if not np.isnan(iou[c]) else None) for c in range(NUM_CLASSES)},
        "mean_iou": round(float(np.nanmean(iou[present])), 4),
        "accuracy": round(float(tp.sum() / total), 4),
        "precision": round(float(precision), 4),
        "recall": round(float(recall), 4),
        "f1": round(float(f1), 4),
        "positive_class": CLASS_NAMES[pos],
        "labelled_pixels": int(total),
        "positive_fraction": round(float(matrix[pos].sum() / total), 4),
        "confusion": matrix.astype(int).tolist(),
    }


def segmentation_metrics(predictions: Sequence[Any], labels: Sequence[Any]) -> dict[str, Any]:
    """Metrics of predicted masks against labels over all chips at once (pixel-pooled, not chip-averaged)."""
    return metrics_from_confusion(confusion_matrix(predictions, labels))


def majority_baseline(labels: Sequence[Any]) -> dict[str, Any]:
    """The all-class-0 prediction scored on the same pixels: the number any model must beat on the positive class."""
    import numpy as np

    predictions = [np.zeros_like(np.asarray(label), dtype=np.int64) for label in labels]
    report = segmentation_metrics(predictions, labels)
    report["note"] = f"predicts '{CLASS_NAMES[0]}' for every pixel"
    return report

**Module 3/3:** `src/prithvi_flood_segmentation_pipeline/samples.py` (carried verbatim; see the note above)

In [ ]:
"""Labelled-chip dataset contract for adapting the flood model: the pinned Sen1Floods11 sample, role assignment
from the official splits, BYOD loaders and sample export.

The default dataset is **real**: 44 hand-labelled 512 × 512 Sentinel-2 chips of Sen1Floods11 (Bonafilia et al.,
2020) — 24 from the official training split, 8 from the validation split and 12 from the test split, drawn
round-robin over the ten flood-event regions with a fixed seed on 2026-09-19 from the chips whose hand label is at
least 60 % valid (not cloud / no-data) and at least 3 % water, so every chip can be scored — pinned here by object
path, byte size and SHA-256 of both the 13-band `S2Hand` GeoTIFF and its `LabelHand` mask. Every object is fetched from the
public Sen1Floods11 bucket at run time and refused on any byte-size or SHA-256 mismatch; the repository
redistributes none of the chips. The roles follow the upstream splits, so the test chips are chips the packaged
model never trained on.

A record is ``{id, image, label}``: a (6, 512, 512) reflectance array (or a GeoTIFF path) and a (512, 512) mask
with 0 = no water, 1 = water, -1 = no data / cloud (or a GeoTIFF path).
"""

from __future__ import annotations

import csv
import hashlib
import io
import json
import urllib.request
import zipfile
from collections.abc import Mapping, Sequence
from pathlib import Path
from typing import Any

# standalone rewrite (build_notebook.py): `from .pipeline import (` removed — names are kernel globals defined by the carried modules

CORPUS_NAME = "Sen1Floods11 hand-labelled Sentinel-2 chips (v1.1)"
CORPUS_RELEASE = "Sen1Floods11 v1.1 public bucket, 44 chips selected 2026-09-19 from the official hand-labelled splits"
CORPUS_BASE_URL = "https://storage.googleapis.com/sen1floods11/v1.1/"
CORPUS_LICENSE = "CC BY 4.0 (Cloud to Street; Bonafilia et al., CVPRW 2020)"
CORPUS_BYTES = 103_757_095
DEFAULT_CACHE_DIR = Path("weights") / "sen1floods11"
ROLES = ("train", "validation", "test")
# (chip name, role from the official split, S2Hand bytes, S2Hand sha256, LabelHand bytes, LabelHand sha256)
SAMPLE_RECORDS: tuple[tuple[str, str, int, str, int, str], ...] = (
    (
        "Ghana_141910",
        "train",
        2287617,
        "9d47e96a72447591200939350171395aa0c0a7c4d25e135d354c90e8889f1d6b",
        3562,
        "e6dd6c0eb8f2bc5e5e64b12e99f7398d27a0e56a4670f8be5382055504c4602b",
    ),
    (
        "Ghana_362274",
        "train",
        2235356,
        "76bea781cdd9ab8dc3992489709d0af14ffb47f2a5ff933b2e678f29db4d7e34",
        4129,
        "2030e61f58a19ed8a58359281959172a044ee4f45e589ff13555a55d9cb90ff9",
    ),
    (
        "Ghana_887131",
        "train",
        2212964,
        "fc41fa6e4187aef78ae0a1469c4ca483c21e09a304cfdd29c24e18a3d59f5fc1",
        5064,
        "f445c0716feb0ca1790b0c9ec3f20b067d36c99324d4632b4332ec2372797138",
    ),
    (
        "India_591549",
        "train",
        2403439,
        "b515458853602595b09331252532e40b38a2798733f8f8f87ceb87d3129dd060",
        8973,
        "ff69850ea4d2557bbbc6dd62bb7e8752725747f156055e431f2cea228209add2",
    ),
    (
        "India_91379",
        "train",
        2307403,
        "e06d91b2a236d8fe30a46ab96708119c9c437d1b40a54b2a766fbcd63efe271d",
        12942,
        "cac872eb9b59c21754d465033cb79cab91b66f833ca8a832d1150bb9acae088b",
    ),
    (
        "India_943439",
        "train",
        2255144,
        "7a3bd870a2e7723d9cbc348a565d974d815c6b999e9de984533e105dcc3bdab1",
        12360,
        "444d84e29ad06cbba1dfdbba1d812ab1918e0ce2884cdfdff0ceb0cb54bc56dc",
    ),
    (
        "Mekong_1396181",
        "train",
        2424141,
        "7441b4ce03633d3bb477d38383b9495081f80619644f4006c8cf0344fbd01be2",
        16633,
        "3cba6ca75f2c277d5e681437ebd8da0e45707e37b82c026b3bc799791ca16438",
    ),
    (
        "Mekong_16233",
        "train",
        2256144,
        "3022128b783154cacbae064f3b1c6c1aa7855d5c9dd91f47d61614a966b33c5e",
        4983,
        "fee52121a2abbd1e1f57f67927122530bf18b7eb336da7ecb2c137f5875f7102",
    ),
    (
        "Mekong_596495",
        "train",
        2411087,
        "7d2adab08af8d576203f17623a3860c6934d1f4079004701351588786f1eb383",
        11426,
        "9fbec9b1bd06e96cc8eb0c66fa4934a48dd80cbd1b670494c2affd259ee7a32c",
    ),
    (
        "Nigeria_529525",
        "train",
        2259657,
        "d84bb2914e73bdb952b820ad63c2c75363a85b2d0c5cf6ff8a5734d33abccc49",
        11861,
        "8a64830e6d1558ec6af8b52b6f4b1fe0ef8920d7f317cfdd8b37bac40f9e8fc9",
    ),
    (
        "Nigeria_600295",
        "train",
        2650879,
        "b418a36e2bd30dbb9452e64e30ec176258ea6a0691630ea980da9d705fa7c9c8",
        12927,
        "7bce532830162964a4d0283046affb74c09f2a1ce3ec5a3c32aa4a6fe2fc629f",
    ),
    (
        "Nigeria_952958",
        "train",
        2448466,
        "49c15c93e8c3e014b7423a6235b0811f83396878831fd23c3d4331611ec54af8",
        6941,
        "f1c8dc26d87e40a4be800f52e5eed855a815f2d1ffeaab476daa7dc729123af9",
    ),
    (
        "Pakistan_246510",
        "train",
        2334917,
        "4115017a9b122d7398822060ab7fe25e55d08543beaa24217ae394d6210eb2e4",
        3328,
        "2ecb22ece2e6ea5c462a9844f1a5f90b813f60366a76eab369b8779ae31be519",
    ),
    (
        "Pakistan_474121",
        "train",
        3172134,
        "37cf02dd20992ade67b071bdde11616a3b1f22ab49e0fb6ec4a8e255edae793c",
        6604,
        "33bae600a8c46dba840a953e338f9e35f4d29f86948cef7dbc20570a0a4966f7",
    ),
    (
        "Paraguay_126224",
        "train",
        1904691,
        "0801c5886ee0072707dd2aeadf99c4c5418a1c40784008cbdda46f1fd122b7fa",
        16766,
        "c8547c0d6031f040bd4e67804eb83339d17e1fd7753b8f18648e9f05f69aeaa8",
    ),
    (
        "Paraguay_822142",
        "train",
        2128900,
        "fdd18e61dfa59fc2c7b6d77823767e558fa84072b0d18f4fb25b8650a5b82c77",
        10588,
        "db2853aeda0cd616ebbe5a7bb5152e78425cd5edcf2f6db53d561f99cfd81d71",
    ),
    (
        "Somalia_1087508",
        "train",
        2579388,
        "d9a9a359407cdd14b96c8561cf9094fe1f02c05fc025bf77497793466797d4d9",
        10949,
        "869ec4bb24f5216b2092e6c8bb8a6a88a18244294212a4053d12771adf3741ee",
    ),
    (
        "Somalia_371421",
        "train",
        2603913,
        "da6858c7fe2d77313efbe849013c9f57c0ce503403929d4ec8d64960f8ca5717",
        12315,
        "f4f16eb1d1c0b74adc2e105ed35d6a457a5695c87980c6b017084fece1a6b9d5",
    ),
    (
        "Spain_2938657",
        "train",
        2364938,
        "ccc542052c18bcb9fc2c97cbc384df01075f30d07f3586761c9329bf22ee2476",
        1991,
        "9d9c2a1bdd688d7665388e8404832908a6b906bcc818f40743bb8ca267f7218f",
    ),
    (
        "Spain_8199661",
        "train",
        2316354,
        "e740ace73679f333f66ea88ae0c252daa030c0fb6489038adb54edb342deda31",
        3618,
        "73b55033d476fbe4744a1589e1cbb71632eef478732eb0819a1b0d8b09e2547b",
    ),
    (
        "Sri-Lanka_653336",
        "train",
        2322882,
        "1ec56dc78347e8e1d81db440ff494ec83d55f7f5b60e7669f6e50bb18a269968",
        5827,
        "a1b69316d33cc9fea53581aa50fcb9608d5fe4b385848e28ca8decd5d79397ad",
    ),
    (
        "Sri-Lanka_883641",
        "train",
        2243119,
        "4f155b3698cb493203762e3e51a0892a0f88909933d93cd174e8d4ffa5efec6a",
        1914,
        "ea7e06bf1badbf2779bd82da0ba9b37a611643d5ebeca1342617f7c525efa930",
    ),
    (
        "USA_1068362",
        "train",
        2131543,
        "82cca7cf72570d3fe2d8ce90c4b36ac326b3561d1dfee7741de98fceb9532f64",
        7009,
        "7cc38de430cf6908daa96a8800857f0b23c6aaa0377fbf04bcc5676f91dd15db",
    ),
    (
        "USA_652955",
        "train",
        2169702,
        "0cf186db362a8f4f414378b1083a25bb915c6ca90c80d126106082e4cc3078d4",
        2764,
        "3d40e98df00c30de3f84b577479c103750d64c5080f3aff7bc70178c798f2755",
    ),
    (
        "Ghana_868803",
        "validation",
        2321183,
        "f31eb46a7ad94dfb6a79cdfa4d6062839d0bd96db620debc64e83e2103d16401",
        7696,
        "b047d8f6d902b44a9c20dc6fce6dbced6b884b8260f1040e61b851050f0b9689",
    ),
    (
        "India_1068117",
        "validation",
        2450753,
        "6a59c927e6f0cabd3206e5303154e6cabe7c2d0669266ffe2649f56a0b39e7ff",
        15733,
        "db5284a5cd91eab5c70eb310350f4e77e861adef6d1daf26244fbfdd1b4c83e0",
    ),
    (
        "Mekong_293769",
        "validation",
        2448110,
        "4d9b10a1e4c7dc05a47d74397c334f25096acc9e1887659b3c0e2986ea6e6a98",
        6528,
        "629304ae517b2c7dd0dc6cb8dd72d5943b4a2943965bd782a7b309f28077ef73",
    ),
    (
        "Nigeria_984831",
        "validation",
        2362185,
        "a7d45a2f8ab4aa6fdd311fe25937e9112cb367fbf7a6cf306b69add29f20846b",
        10322,
        "99ac70c34aaca2ce48a157f247ef35e7eb1c57af3a58d5f6236e74b3156eed94",
    ),
    (
        "Pakistan_1027214",
        "validation",
        2245598,
        "c78cc47a8eeae63dfe57eb426dbe01c8d9b7f364d07889b0ce55d607869e2c85",
        10004,
        "82f1a0c7677d4b14ebda580884a6e8e854c2bf00ba5f470c439456e3da2c1902",
    ),
    (
        "Paraguay_581976",
        "validation",
        1914206,
        "aa0bf782e01fb60b002aa5ae2397d73fe07861d3ea8cc7a3aba9ffe7c00a6c5f",
        15614,
        "40035afa001ba84d7b59dd593fa2fa6743b1d4af1356b4f478046b734b870ad9",
    ),
    (
        "Somalia_12849",
        "validation",
        2617333,
        "43781ffe14ff9604d0538ec9a6f7c49ffb9ae488b0f08ffa3dcbbb1dc207b7d9",
        15222,
        "9b7cb1c8a378c193b1be9405c6ea200c38c3fc80300d190184c0226383c0afcb",
    ),
    (
        "Spain_1199913",
        "validation",
        2348104,
        "f76d9be2ab682c7c88fcaa3bf621aa19fa697fc82b848c1fe2034d0c36be5138",
        2348,
        "27fedadf6819805196eb1061a21e11ccb7ba26648a2424917b9de3174d45f2e0",
    ),
    (
        "Ghana_313799",
        "test",
        2243083,
        "621f5c19765e6e6a28d3a3b8bb357188ecb4d65ab4f14dba38e30bd84b93ce40",
        6073,
        "3bde81b5172d5017ec561ce137cdf8d59fbf3a69fc7796d08188b97219a32862",
    ),
    (
        "Ghana_319168",
        "test",
        2552380,
        "2a46104f1a75903ce2b0dd98b82e3e20236ea997071c6e4e801af903ad9d0445",
        11209,
        "da46da70d9b18f8851a58c3868a4a4b6262ebfa4433ff9306feaa101b634f33e",
    ),
    (
        "India_592446",
        "test",
        2418725,
        "4a041b3f3f28e5219da53c56d3715bc7a75925b235b69a5d1628951fd9e5805c",
        5708,
        "23cf1fcd9882fac00552bc25ec4202e95d58421eff597c9cf659c4f733c49cd0",
    ),
    (
        "India_747992",
        "test",
        2341250,
        "4e9cf7dff2a8b6495b18e4abf12cfe587a4b3d28deadccfe4354b60b1f40c9ae",
        17053,
        "c63a63987d65748a87ddd4c7685c7334eaa37955ce4e4cad6ce26af764bab8a9",
    ),
    (
        "Mekong_382276",
        "test",
        2268452,
        "3065622535054f19f5c517dcfe2308dea0a71825e2a86479d914be4932c9b31f",
        2201,
        "9ff60aa49161aec241227fd6e9fed2c89a6f22ec839ef9dbf76268d399996631",
    ),
    (
        "Nigeria_812045",
        "test",
        2433122,
        "14053b65f4f4d453c1cc7d7a4ff7b0a1124880f81404985d798ff616364699b6",
        9767,
        "77386f8586a6a825430b11d74619fd39391f2c5d3fc2fc93d829bc15f4933308",
    ),
    (
        "Pakistan_849790",
        "test",
        2179387,
        "facb36e9d7ad50bce80d8af8be7a95af3ad20c696b77896120db86d460cc8dce",
        17848,
        "669bcd99f5cedcf37db82c7eff7dc0c20036588eb1462321cf8fecf727945586",
    ),
    (
        "Paraguay_868895",
        "test",
        2511881,
        "a33f701211cd8a509e2c1428378d2e88039d162a8f1fcab68bf682882aa200e6",
        10893,
        "c4da12e7aa9eb2a42f3594c9744f63322b2ea6cdb0ce17963bfb114452e5d3c1",
    ),
    (
        "Somalia_94102",
        "test",
        2574036,
        "a7b40d8260e5e3f055c42e8e6c661f47fcb78a4b90b21d5522763ba335691b7a",
        6569,
        "5177ba119333f3d31a94c18f113bfe593925286dacbbb86780ff664b0ca49681",
    ),
    (
        "Spain_6095801",
        "test",
        2319727,
        "a21c5574a35f058bb3e3cefb1d6625b7a156bde8b38ad2b34ef09685863a01c9",
        7549,
        "f0c9a7ad9c7049ca43363580eb8c6849812b51bb8095901fcc2796400817f512",
    ),
    (
        "Sri-Lanka_377277",
        "test",
        2200577,
        "3caae5b0815a852bc6c8dae352c0906d13c2d63c1f04e8bc7d94537c94c4bf2d",
        3456,
        "6540085bae0fa58de10f7b6e0cdbb1df3498c2276b8c7b7f14628759c7cbc063",
    ),
    (
        "USA_430764",
        "test",
        2199014,
        "385418e105dc1068a7d78585e4395bcdc134e7b0d092ba942867ef74393d8d12",
        5944,
        "ae84a33d041a947fc20d36bdeb2176d80f50bc5161dafa4b372df9f9d75ebe50",
    ),
)
SAMPLE_LABEL_SOURCE = f"{CORPUS_NAME}; {CORPUS_RELEASE}; {CORPUS_LICENSE}"


def _sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


def object_path(name: str, kind: str) -> str:
    """Bucket-relative path of a chip's image (`S2Hand`) or label (`LabelHand`) object."""
    folder = {"image": "S2Hand", "label": "LabelHand"}[kind]
    return f"data/flood_events/HandLabeled/{folder}/{name}_{folder}.tif"


def fetch_object(rel: str, *, expected: tuple[int, str], cache_dir: str | Path | None = None, fetcher: Any = None) -> bytes:
    """Return one pinned object's bytes from the cache or the bucket, refused on a size or digest mismatch."""
    cache = Path(cache_dir) if cache_dir is not None else DEFAULT_CACHE_DIR
    cache.mkdir(parents=True, exist_ok=True)
    local = cache / Path(rel).name
    size, digest = expected
    data = local.read_bytes() if local.is_file() else b""
    if len(data) != size or _sha256_bytes(data) != digest:
        url = CORPUS_BASE_URL + rel
        if fetcher is not None:
            data = fetcher(url)
        else:
            request = urllib.request.Request(url, headers={"User-Agent": "dimer-prithvi-flood-tutorial/1.0"})
            with urllib.request.urlopen(request, timeout=120) as response:  # noqa: S310 (pinned https URL)
                data = response.read()
        if len(data) != size or _sha256_bytes(data) != digest:
            raise ValueError(
                f"{rel}: fetched {len(data)} bytes with sha256 {_sha256_bytes(data)[:16]}…, pinned {size} / {digest[:16]}…"
            )
        local.write_bytes(data)
    return data


def fetch_corpus(*, cache_dir: str | Path | None = None, fetcher: Any = None) -> dict[str, dict[str, bytes]]:
    """Every pinned chip's image and label bytes, keyed by chip name."""
    out = {}
    for name, _role, image_bytes, image_sha, label_bytes, label_sha in SAMPLE_RECORDS:
        out[name] = {
            "image": fetch_object(
                object_path(name, "image"), expected=(image_bytes, image_sha), cache_dir=cache_dir, fetcher=fetcher
            ),
            "label": fetch_object(
                object_path(name, "label"), expected=(label_bytes, label_sha), cache_dir=cache_dir, fetcher=fetcher
            ),
        }
    return out


def read_corpus(files: Mapping[str, Mapping[str, bytes]]) -> dict[str, list[dict[str, Any]]]:
    """Decode the verified bytes into `{id, image, label}` records grouped by role (train / validation / test)."""
    import tempfile

    splits: dict[str, list[dict[str, Any]]] = {role: [] for role in ROLES}
    for name, role, *_ in SAMPLE_RECORDS:
        if name not in files:
            raise ValueError(f"corpus is missing {name}")
        with tempfile.TemporaryDirectory() as tmp:
            image_path = Path(tmp) / "image.tif"
            label_path = Path(tmp) / "label.tif"
            image_path.write_bytes(files[name]["image"])
            label_path.write_bytes(files[name]["label"])
            image = read_chip(image_path, band_indices=S2_L1C_BAND_INDICES)
            label = read_mask(label_path)
        raw = {
            "id": f"{role}-{len(splits[role]):03d}",
            "source_id": name,
            "region": name.split("_")[0],
            "split": role,
            "image": image,
            "label": label,
            "source": f"{CORPUS_BASE_URL}{object_path(name, 'image')}",
        }
        splits[role].append(check_record(raw))  # scaled to reflectance, no-data replaced, label checked
    return splits


def fetch_sample_dataset(*, cache_dir: str | Path | None = None, fetcher: Any = None) -> dict[str, list[dict[str, Any]]]:
    """The tutorial splits from the pinned corpus (roles from the official Sen1Floods11 splits)."""
    return read_corpus(fetch_corpus(cache_dir=cache_dir, fetcher=fetcher))


def check_split_disjoint(splits: Mapping[str, Sequence[Mapping[str, Any]]]) -> dict[str, Any]:
    """Assert no chip (by pixel digest) appears in two splits (leakage check)."""
    seen: dict[str, str] = {}
    for name, records in splits.items():
        for record in records:
            key = chip_digest(record)
            if key in seen and seen[key] != name:
                raise ValueError(f"chip {record['id']!r} appears in both {seen[key]} and {name}")
            seen[key] = name
    return {name: len(records) for name, records in splits.items()}


def split_dataset(
    records: Sequence[Mapping[str, Any]],
    *,
    val_fraction: float = 0.2,
    test_fraction: float = 0.25,
    seed: int = 0,
) -> dict[str, list[dict[str, Any]]]:
    """Seeded shuffle of a BYOD dataset into train / validation / test after de-duplicating chips. Chips from one
    scene or event are near-duplicates; group them yourself (one region per split) when that matters."""
    import random

    if not (0.0 <= val_fraction < 1.0 and 0.0 < test_fraction < 1.0 and val_fraction + test_fraction < 1.0):
        raise ValueError("fractions must satisfy 0 <= val < 1, 0 < test < 1, val + test < 1")
    checked = validate_dataset(records)["records"]
    seen: set[str] = set()
    unique = []
    for record in checked:
        key = chip_digest(record)
        if key not in seen:
            seen.add(key)
            unique.append(record)
    rng = random.Random(seed)
    rng.shuffle(unique)
    n_test = max(1, round(len(unique) * test_fraction))
    n_val = round(len(unique) * val_fraction)
    splits = {"test": unique[:n_test], "validation": unique[n_test : n_test + n_val], "train": unique[n_test + n_val :]}
    if len(splits["train"]) < MIN_RECORDS:
        raise ValueError(f"split leaves {len(splits['train'])} training chips; at least {MIN_RECORDS} are required")
    return splits


def load_byod_dataset(path: str | Path) -> list[dict[str, Any]]:
    """Read `{id, image, label}` records from a directory or a zip holding `pairs.csv` (columns `id`, `image`,
    `label`) beside six-band 512 × 512 GeoTIFF chips and single-band label rasters; files are decoded from bytes,
    never extracted to disk."""
    import tempfile

    source = Path(path)
    if source.is_dir():
        table = (source / "pairs.csv").read_text(encoding="utf-8")
        loader = lambda name: (source / name).read_bytes()  # noqa: E731
    elif source.is_file() and source.suffix.lower() == ".zip":
        archive = zipfile.ZipFile(source)
        members = {Path(n).name: n for n in archive.namelist()}
        if "pairs.csv" not in members:
            raise ValueError("BYOD zip must contain pairs.csv")
        table = archive.read(members["pairs.csv"]).decode("utf-8")
        loader = lambda name: archive.read(members[name])  # noqa: E731
    else:
        raise ValueError("BYOD datasets must be a directory or a .zip holding pairs.csv and the GeoTIFF files")
    rows = list(csv.DictReader(io.StringIO(table)))
    missing = {"id", "image", "label"} - set(rows[0].keys() if rows else set())
    if missing:
        raise ValueError(f"pairs.csv is missing columns {sorted(missing)}")
    out = []
    with tempfile.TemporaryDirectory() as tmp:
        for row in rows:
            image_path = Path(tmp) / "image.tif"
            image_path.write_bytes(loader(row["image"]))
            record: dict[str, Any] = {"id": row["id"], "image": read_chip(image_path)}
            if row.get("label"):
                label_path = Path(tmp) / "label.tif"
                label_path.write_bytes(loader(row["label"]))
                record["label"] = read_mask(label_path)
            out.append(record)
    return out


def write_sample_pair(record: Mapping[str, Any], image_path: str | Path, label_path: str | Path) -> dict[str, str]:
    """Write one record as a six-band float32 TIFF and a single-band int16 TIFF (the BYOD shape, without
    georeferencing) and return both paths."""
    import numpy as np
    import tifffile

    image_out, label_out = Path(image_path), Path(label_path)
    image_out.parent.mkdir(parents=True, exist_ok=True)
    tifffile.imwrite(image_out, np.asarray(record["image"], dtype=np.float32), photometric="minisblack", planarconfig="separate")
    tifffile.imwrite(label_out, np.asarray(record["label"], dtype=np.int16), photometric="minisblack")
    return {"image": str(image_out), "label": str(label_out)}


def write_dataset_csv(records: Sequence[Mapping[str, Any]], path: str | Path) -> Path:
    """Write the pairs table of a split (id, image, label, provenance) in the shape BYOD expects."""
    out = Path(path)
    out.parent.mkdir(parents=True, exist_ok=True)
    with open(out, "w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=["id", "image", "label", "region", "source"])
        writer.writeheader()
        for record in records:
            writer.writerow(
                {
                    "id": record["id"],
                    "image": f"{record.get('source_id', record['id'])}_S2Hand.tif",
                    "label": f"{record.get('source_id', record['id'])}_LabelHand.tif",
                    "region": record.get("region", ""),
                    "source": record.get("source", ""),
                }
            )
    return out


def dataset_manifest(splits: Mapping[str, Sequence[Mapping[str, Any]]]) -> dict[str, Any]:
    """Validate every split and summarise the dataset (counts, class balance, digests) for provenance exports."""
    summary: dict[str, Any] = {"model_id": MODEL_ID, "image_size": IMAGE_SIZE, "splits": {}}
    for name, records in splits.items():
        report = validate_dataset(records, min_records=1)
        summary["splits"][name] = {
            "n_records": report["n_records"],
            "class_pixel_fraction": report["class_pixel_fraction"],
            "ignored_pixels": report["ignored_pixels"],
            "regions": sorted({str(r.get("region", "")) for r in records if r.get("region")}),
            "digest": report["digest"],
        }
    summary["disjoint"] = check_split_disjoint(splits)
    digests = json.dumps({k: v["digest"] for k, v in summary["splits"].items()}, sort_keys=True)
    summary["digest"] = hashlib.sha256(digests.encode("utf-8")).hexdigest()
    return summary

## 3. Pin, stage and verify the model

The model identity is carried twice — `MODEL_ID`/`MODEL_REVISION` in the module above and the `7`-file manifest below (paths, byte sizes, SHA-256) — and the cell first asserts they agree. It writes the manifest into the working-directory snapshot, then `stage_missing_files(..., allow_download=True)` fetches exactly the entries that are absent from the Hugging Face Hub **at revision `91ce9d38086a…`** (never `main`), `verify_snapshot` re-hashes every file and raises on the first size or digest mismatch, and only then does `PrithviFloodPipeline.from_pretrained(weights_dir=WEIGHTS_DIR, device=('cuda' if torch.cuda.is_available() else 'cpu'), report=print)` load the verified files. There is no fallback to a different download and no remote model code is executed. The effective identity, device and weight source are printed before any inference.

In [ ]:
import json

MANIFEST = {
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "prithvi-eo-2.0-300m-tl-sen1floods11",
  "modelId": "ibm-nasa-geospatial/Prithvi-EO-2.0-300M-TL-Sen1Floods11",
  "revision": "91ce9d38086a80b078a192b374df758b8855b732",
  "files": [
    {
      "path": "README.md",
      "bytes": 3243,
      "sha256": "2aff52e3b680ca1f0b81a546dc733b4735a8611ab9b9623a3b0747d3e70c03cd"
    },
    {
      "path": "config.json",
      "bytes": 3365,
      "sha256": "f03dee42a0cb6433d0e2b8dd109f852948142b221582ce06db97dc5e932afd62"
    },
    {
      "path": "config.yaml",
      "bytes": 3668,
      "sha256": "ca2fa45885b85afac9400c57f7ca3b711526cbad3af285f8b67de74ba2db404a"
    },
    {
      "path": "Prithvi-EO-V2-300M-TL-Sen1Floods11.pt",
      "bytes": 1276843350,
      "sha256": "76eed77d8bd543ae441308b80e8408502243a871cb8a338ddc8220ed96dfc270"
    },
    {
      "path": "examples/India_900498_S2Hand.tif",
      "bytes": 2151620,
      "sha256": "ee898621b387a731503a01960397599f209b45c268d4e088689928c704dbe968"
    },
    {
      "path": "examples/Spain_7370579_S2Hand.tif",
      "bytes": 2320690,
      "sha256": "16e997e6a7159fa11160faf00591da763eb37ac82faf806f7fe733991944a048"
    },
    {
      "path": "examples/USA_430764_S2Hand.tif",
      "bytes": 2199014,
      "sha256": "385418e105dc1068a7d78585e4395bcdc134e7b0d092ba942867ef74393d8d12"
    }
  ],
  "totalBytes": 1283524950
}

if (MANIFEST['modelId'], MANIFEST['revision']) != (MODEL_ID, MODEL_REVISION):
    raise RuntimeError('inline manifest does not name the identity carried by the pipeline module; the notebook was not regenerated after a change')
WEIGHTS_DIR = DEFAULT_WEIGHTS_DIR
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
with open(WEIGHTS_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(MANIFEST, handle, indent=2)
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'license': MODEL_LICENSE, 'files': len(MANIFEST['files']), 'total_bytes': MANIFEST['totalBytes']})
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(WEIGHTS_DIR), 'fetched': fetched})
snapshot = verify_snapshot(WEIGHTS_DIR)
_files = snapshot.get('files', []) if isinstance(snapshot, dict) else []
print({'verified_files': len(_files) if isinstance(_files, list) else _files, 'revision': snapshot.get('revision', MODEL_REVISION) if isinstance(snapshot, dict) else MODEL_REVISION})
pipe = PrithviFloodPipeline.from_pretrained(weights_dir=WEIGHTS_DIR, device=('cuda' if torch.cuda.is_available() else 'cpu'), report=print)
print({'device': getattr(pipe, 'device', None), 'source': getattr(pipe, 'source', 'local-snapshot')})

## 4. Sample chips, validation and roles

The default dataset is 44 hand-labelled Sentinel-2 chips of Sen1Floods11 — 24 from the official training split, 8 from the validation split and 12 from the test split, drawn across the ten flood-event regions from the chips whose label is at least 60 % valid and 3 % water — fetched by object path from the public bucket and refused on any byte-size or SHA-256 mismatch (`fetch_corpus`). Each 13-band `S2Hand` GeoTIFF is reduced to the six Prithvi bands, scaled from reflectance × 10 000 to reflectance, and its no-data replaced by 0, exactly as the upstream datamodule does; each `LabelHand` mask keeps −1 for cloud / no-data, which every metric ignores. `dataset_manifest` validates every split, checks that no chip appears twice and records a digest.

Look for: 24 / 8 / 12 chips with water fractions around 0.2..0.4, a written sample pair (`outputs/prithvi_flood_segmentation_sample_chip.tif` + `_sample_label.tif`, the BYOD shape), and three refusal probes — a five-band chip, a label with an unknown class, a chip with reflectance far outside range — each rejected before the model runs.

In [ ]:
import json
import os
from pathlib import Path

import numpy as np

USE_BYOD = False  # @param {type:"boolean"}

os.makedirs('outputs', exist_ok=True)
if USE_BYOD:
    from google.colab import files
    uploaded = files.upload()
    file_name, payload = next(iter(uploaded.items()))
    byod_path = Path('work') / file_name
    byod_path.parent.mkdir(parents=True, exist_ok=True)
    byod_path.write_bytes(payload)
    splits = split_dataset(load_byod_dataset(byod_path), seed=0)
    data_source = 'BYOD (' + file_name + ')'
else:
    splits = fetch_sample_dataset(cache_dir='weights/sen1floods11')
    data_source = SAMPLE_LABEL_SOURCE
train_records, val_records, test_records = splits['train'], splits['validation'], splits['test']

dataset_report = dataset_manifest({'train': train_records, 'validation': val_records, 'test': test_records})
print({'data_source': data_source, 'splits': {k: v['n_records'] for k, v in dataset_report['splits'].items()}, 'disjoint': dataset_report['disjoint'], 'digest': dataset_report['digest'][:16] + '...'})
for name, part in dataset_report['splits'].items():
    print({name: {'water_fraction': part['class_pixel_fraction']['water'], 'ignored_pixels': part['ignored_pixels'], 'regions': part['regions']}})
print({'first_test_chip': validate_inputs(test_records[0])})
sample_pair = write_sample_pair(test_records[0], 'outputs/prithvi_flood_segmentation_sample_chip.tif', 'outputs/prithvi_flood_segmentation_sample_label.tif')
print({'sample_pair': sample_pair, 'pairs_csv': str(write_dataset_csv(test_records, 'outputs/prithvi_flood_segmentation_sample_pairs.csv'))})

print({'validation': INPUT_SCHEMA['validation']})
probes = {
    'five-band chip': [{**test_records[0], 'image': test_records[0]['image'][:5]}, *test_records[1:4]],
    'unknown label class': [{**test_records[0], 'label': np.where(test_records[0]['label'] == 1, 7, test_records[0]['label'])}, *test_records[1:4]],
    'reflectance out of range': [{**test_records[0], 'image': test_records[0]['image'] * 50000.0}, *test_records[1:4]],
}
for name, records in probes.items():
    try:
        validate_dataset(records)
        print({'probe': name, 'verdict': 'accepted'})
    except (TypeError, ValueError) as exc:
        print({'probe': name, 'rejected': str(exc)[:110]})

## 5. The frozen model against the no-water baseline

`pipe.predict` standardises each chip with the upstream datamodule's band statistics, runs the encoder, neck, decoder and head in float16 autocast, and returns the argmax mask, the softmax scores (the model's outputs, not calibrated probabilities) and the water fraction per chip. `pipe.evaluate` pools the labelled pixels of every held-out chip into one confusion matrix (−1 pixels excluded) and reports the per-class IoU, mean IoU, accuracy, and the water class's precision, recall and F1; the **no-water baseline** — every pixel predicted as land — is scored on the same pixels, so its accuracy is exactly the land fraction and its water IoU is 0.

Look for: a water IoU well above 0 on the test chips (in the build record about 0.59, with recall above precision — the model over-predicts water on some regions), a validation water IoU around 0.86, and per-chip water fractions that track the labels. These are sample-sanity numbers on 12 and 8 chips, not the benchmark.

In [ ]:
import time

t0 = time.perf_counter()
frozen_test = pipe.evaluate(test_records)
frozen_val = pipe.evaluate(val_records)
print({'seconds': round(time.perf_counter() - t0, 1), 'metric': frozen_test['metric']})
print({'baseline_no_water_test': {k: frozen_test['baseline_no_water'][k] for k in ('iou', 'accuracy', 'f1')}})
print({'frozen_test': {k: frozen_test['model'][k] for k in ('iou', 'mean_iou', 'accuracy', 'precision', 'recall', 'f1')}})
print({'frozen_validation': {k: frozen_val['model'][k] for k in ('iou', 'f1')}})
frozen_predictions = pipe.predict(test_records)
for record, pred in list(zip(test_records, frozen_predictions['predictions']))[:6]:
    labelled = record['label'] >= 0
    print({'chip': record['source_id'], 'water_label': round(float((record['label'] == 1).sum() / labelled.sum()), 3), 'water_predicted': pred['class_fraction']['water'], 'ignored': int((~labelled).sum())})
print({'decision_rule': frozen_predictions['decision_rule'], 'scores_shape': frozen_predictions['predictions'][0]['scores'].shape})

## 6. Bounded fine-tuning of the neck, decoder and head

`pipe.adapt` trains the 46 tensors of the pyramid neck, the UPerNet decoder and the head (15.1 M parameters — 4.7 % of the model) and nothing else: the ViT-L encoder is frozen (no gradient is stored for it), and every BatchNorm layer keeps its running statistics, because batches of two chips would corrupt them. Each step takes two chips with a seeded horizontal or vertical flip, computes the cross-entropy over the labelled pixels (−1 ignored) and takes an AdamW step at a small fixed learning rate with gradient-norm clipping and float16 loss scaling. Epoch 0 records the frozen model's validation loss and metrics; the epoch with the lowest validation loss is kept — which can be epoch 0, since the packaged model already trained on this dataset.

Watch the validation loss: in the build record it dipped at epoch 2 and rose again by epoch 4 — the sign that a small learning rate and validation selection are doing their job on a model that has little left to learn from 24 chips it has already seen the distribution of. Four epochs (48 steps) take under a minute on a T4. `TRAINABLE = 'decoder+last_block'` also unfreezes the last encoder block (27.7 M parameters).

In [ ]:
EPOCHS = 4  # @param {type:"integer"}
LEARNING_RATE = 1e-5  # @param {type:"number"}
BATCH_SIZE = 2  # @param {type:"integer"}
TRAINABLE = 'decoder'  # @param ["decoder", "decoder+last_block"]

def report(entry):
    row = {'epoch': entry['epoch'], 'train_loss': None if entry['train_loss'] is None else round(entry['train_loss'], 4), 'val_loss': round(entry['val_loss'], 4)}
    if 'val' in entry:
        row['val_water_iou'] = entry['val']['iou']['water']
        row['val_f1'] = entry['val']['f1']
    if 'note' in entry:
        row['note'] = entry['note']
    print(row)

t0 = time.perf_counter()
adapt_result = pipe.adapt(train_records, val_records, epochs=EPOCHS, lr=LEARNING_RATE, batch_size=BATCH_SIZE, trainable=TRAINABLE, progress=report)
adapt_seconds = round(time.perf_counter() - t0, 1)
print({'trainable_parameters': adapt_result['n_trainable'], 'total_parameters': adapt_result['n_total'], 'steps': adapt_result['n_steps'], 'best_epoch': adapt_result['best_epoch'], 'precision': adapt_result['precision'], 'batchnorm': adapt_result['batchnorm'], 'seconds': adapt_seconds})

## 7. Held-out evaluation: the paired comparison

The test chips were never used for training or epoch selection (they come from the official test split). The adapted model is scored exactly as the frozen model was in Section 5, and the table puts the baseline, the frozen and the adapted numbers side by side. The cell asserts what the procedure guarantees — the kept epoch's validation loss is no higher than the frozen model's — and prints the test numbers without asserting a direction: on this sample the water IoU moved from about 0.59 to 0.60 in the build record, a sample-sanity observation on 12 chips with no dispersion estimate, not a quality claim. Twelve chips from seven regions cannot separate a real gain from noise; with your own chips from a new sensor or region, the gap between frozen and adapted is the number to watch.

In [ ]:
adapted_test = pipe.evaluate(test_records)
adapted_val = pipe.evaluate(val_records)
comparison = {}
for key in ('mean_iou', 'accuracy', 'precision', 'recall', 'f1'):
    comparison[key] = {'baseline_no_water': frozen_test['baseline_no_water'][key], 'frozen': frozen_test['model'][key], 'adapted': adapted_test['model'][key]}
comparison['water_iou'] = {'baseline_no_water': frozen_test['baseline_no_water']['iou']['water'], 'frozen': frozen_test['model']['iou']['water'], 'adapted': adapted_test['model']['iou']['water']}
for key, row in comparison.items():
    print({key: row})
print({'validation_water_iou': {'frozen': frozen_val['model']['iou']['water'], 'adapted': adapted_val['model']['iou']['water']}, 'validation_loss': {'frozen': adapt_result['history'][0]['val_loss'], 'kept_epoch': adapt_result['history'][adapt_result['best_epoch']]['val_loss']}})
evaluation_report = {
    'model': {'id': MODEL_ID, 'revision': MODEL_REVISION, 'key': MODEL_KEY},
    'data_source': data_source,
    'dataset': dataset_report,
    'frozen': {'test': frozen_test, 'validation': frozen_val},
    'adapted': {'test': adapted_test, 'validation': adapted_val},
    'comparison': comparison,
    'adaptation': {k: v for k, v in adapt_result.items() if k not in ('history', 'trainable_names')},
    'history': adapt_result['history'],
    'adaptation_seconds': adapt_seconds,
}
with open('outputs/prithvi_flood_segmentation_evaluation_report.json', 'w', encoding='utf-8') as f:
    json.dump(evaluation_report, f, indent=2)
assert adapt_result['history'][adapt_result['best_epoch']]['val_loss'] <= adapt_result['history'][0]['val_loss']
assert adapted_val['model'] == adapt_result['history'][adapt_result['best_epoch']]['val']
print({'report': 'outputs/prithvi_flood_segmentation_evaluation_report.json'})

## 8. New chips, artifact export and fresh reload

The adapted model segments the three example chips that ship with the upstream repository (India, Spain, USA — 13-band Sentinel-2 L1C files reduced to the six bands), which carry no labels here: the predicted water fraction per chip and a written mask are a sanity check, not an evaluation.

`pipe.save_artifact` writes the trained tensors (about 60 MB) as `adapter.safetensors`, with a `manifest.json` recording the artifact format, the base model id and revision, the digest of the converted base file, the adaptation scope, the tensor names, the file size and SHA-256, the training configuration and the epoch history (OUT8). `PrithviFloodPipeline.from_artifact` re-verifies the base file, checks the artifact manifest, scope and digest **before** deserialising, rebuilds the model and overlays the tensors — a fresh object from files, not the in-memory model (VER2). The cell asserts identical held-out metrics and identical score maps (VER4).

In [ ]:
import platform
import shutil

import tifffile

example_dir = WEIGHTS_DIR / 'examples'
new_records = [{'id': path.stem, 'image': read_chip(path, band_indices=S2_L1C_BAND_INDICES), 'source': str(path.name)} for path in sorted(example_dir.glob('*.tif'))]
new_predictions = pipe.predict(new_records)
for record, pred in zip(new_records, new_predictions['predictions']):
    tifffile.imwrite(f'outputs/prithvi_flood_segmentation_mask_' + record['id'] + '.tif', pred['mask'])
    print({'chip': record['id'], 'water_fraction': pred['class_fraction']['water'], 'note': 'sanity check, no label'})
with open('outputs/prithvi_flood_segmentation_predictions.json', 'w', encoding='utf-8') as f:
    json.dump({'model': new_predictions['model'], 'classes': new_predictions['classes'], 'decision_rule': new_predictions['decision_rule'], 'predictions': [{'id': p['id'], 'class_fraction': p['class_fraction']} for p in new_predictions['predictions']]}, f, indent=2)

artifact_dir = Path('outputs/prithvi_flood_segmentation_adapter')
shutil.rmtree(artifact_dir, ignore_errors=True)
pipe.save_artifact(artifact_dir, metadata={'tutorial': 'prithvi_flood_segmentation', 'data_source': data_source})
artifact_manifest = json.loads((artifact_dir / 'manifest.json').read_text(encoding='utf-8'))
print({'artifact': str(artifact_dir), 'format': artifact_manifest['format'], 'trainable': artifact_manifest['adapter']['trainable'], 'tensors': len(artifact_manifest['tensors']), 'bytes': artifact_manifest['files'][0]['bytes'], 'sha256': artifact_manifest['files'][0]['sha256'][:16] + '...'})

reloaded = PrithviFloodPipeline.from_artifact(artifact_dir, weights_dir=WEIGHTS_DIR, device=pipe.device)
reloaded_test = reloaded.evaluate(test_records)
before = pipe.predict(test_records[:2])['predictions']
after = reloaded.predict(test_records[:2])['predictions']
parity = {'metrics_identical': reloaded_test['model'] == adapted_test['model'], 'max_abs_score_diff': max(float(np.abs(a['scores'] - b['scores']).max()) for a, b in zip(before, after))}
print({'reload_parity': parity, 'reloaded_best_epoch': reloaded.adapter['best_epoch']})
assert parity['metrics_identical'] and parity['max_abs_score_diff'] < 1e-4

result_payload = {
    'notebook_source': NOTEBOOK_SOURCE,
    'repository_revision': NOTEBOOK_SOURCE['repository_revision'],
    'model': {**evaluation_report['model'], 'model_license': MODEL_LICENSE, 'device': pipe.device, 'source': pipe.source},
    'provenance': {
        'source_asset': [e for e in MANIFEST['files'] if e['path'] == SOURCE_CKPT_NAME],
        'pickle_audit_sha256': PICKLE_AUDIT_SHA256,
        'converted': verify_converted(WEIGHTS_DIR)['files'],
        'pickle_unpickled_once_for_conversion': True,
        'served_from_pickle': False,
        'remote_code_executed': False,
        'data_objects': 2 * len(SAMPLE_RECORDS),
        'data_base_url': CORPUS_BASE_URL,
        'data_license': CORPUS_LICENSE,
    },
    'runtime': {'python': platform.python_version(), 'torch': torch.__version__, 'timm': timm.__version__, 'lightning': lightning.__version__, 'tifffile': tifffile.__version__, 'terratorch': importlib.metadata.version('terratorch')},
    'data_source': data_source,
    'comparison': comparison,
    'artifact': {'dir': str(artifact_dir), 'sha256': artifact_manifest['files'][0]['sha256'], 'bytes': artifact_manifest['files'][0]['bytes']},
    'reload_parity': parity,
}
with open('outputs/prithvi_flood_segmentation_result.json', 'w', encoding='utf-8') as f:
    json.dump(result_payload, f, indent=2)

print('outputs/:')
for path in sorted(Path('outputs').rglob('*')):
    if path.is_file():
        print(f'  - {path.as_posix()} ({path.stat().st_size / 1024:.1f} KB)')

## Interpretation and limits

On 12 held-out chips from the official test split the packaged flood model finds water with an IoU near 0.6 and a recall above its precision, against a no-water baseline that scores 0; a bounded fine-tuning of its neck, decoder and head on 24 chips, selected by validation loss with the frozen model as a candidate, leaves those numbers about where they were. That is the claim: the adaptation contract runs end to end on real labelled multispectral chips, the pickle is audited and converted rather than served, and the artifact that carries the change is 60 MB and reloads with identical outputs. It is not a claim that this sample improves the model — the model already trained on this dataset — nor that 12 chips measure its skill.

The numbers are sample-sanity evidence: one seeded run, 12 test chips from seven regions, no dispersion estimate, pixel-pooled metrics that let large chips dominate, and hand labels that carry their own uncertainty at water edges and under thin cloud. Nothing here measures the model on Sentinel-1, on scenes larger than a chip, or on regions and seasons outside Sen1Floods11.

Three things to carry to real data. **The six bands and their scaling are the contract:** blue, green, red, narrow NIR, SWIR 1, SWIR 2 in that order, reflectance in [0, 1]; a different band order or an uncorrected product is segmented without complaint and silently wrong. **Split by scene or event, not by chip:** neighbouring chips are near-duplicates, and a random split makes memorisation look like skill. **Read the baseline first:** on a chip with 5 % water the no-water baseline is 95 % accurate; only the water IoU, precision and recall say whether the model did anything.

Successful execution proves that the recorded repository revision's pipeline modules, carried in this standalone notebook, can acquire and digest-verify a pickled upstream checkpoint, audit and convert it into safetensors without executing anything outside the audited allow-list, rebuild the model from the installed package, fetch and validate digest-pinned real labelled chips, execute bounded fine-tuning, evaluate against a baseline and the frozen model on held-out chips, and emit the shown machine-readable artifacts — without the repository being reachable. It does **not** establish benchmark superiority, production fitness, or flood-mapping skill beyond the checks shown.

**Optional experiments (they do not affect the default path):** set `TRAINABLE = 'decoder+last_block'`; raise `EPOCHS` and watch the validation loss turn; try `LEARNING_RATE = 1e-4` to see the frozen model win every epoch; or bring your own labelled chips through BYOD and read the baseline before the adapted number.

## References

- Repository README: https://github.com/kurtvalcorza/prithvi-flood-segmentation-pipeline/blob/main/README.md
- Repository model card: https://github.com/kurtvalcorza/prithvi-flood-segmentation-pipeline/blob/main/MODEL_CARD.md
- Weights and conversion notes: https://github.com/kurtvalcorza/prithvi-flood-segmentation-pipeline/blob/main/docs/WEIGHTS.md
- Hugging Face model repository: https://huggingface.co/ibm-nasa-geospatial/Prithvi-EO-2.0-300M-TL-Sen1Floods11 (revision `91ce9d38086a80b078a192b374df758b8855b732`)
- Szwarcman, D., Roy, S., Fraccaro, P., et al. (2024). Prithvi-EO-2.0: A versatile multi-temporal foundation model for Earth observation applications. arXiv:2412.02732: https://arxiv.org/abs/2412.02732
- Bonafilia, D., Tellman, B., Anderson, T., Issenberg, E. (2020). Sen1Floods11: A georeferenced dataset to train and test deep learning flood algorithms for Sentinel-1. CVPR Workshops: https://github.com/cloudtostreet/Sen1Floods11
- TerraTorch: https://github.com/IBM/terratorch
- DIMER Notebook Specification 2.0 and Model Card Specification 1.1 (fleet specs in the ml-worker repository)